# MangoLeaf AI V5.1.0: fresh ImageNet training with expanded Outlier Exposure

Unexecuted experimental template. Train a new seven-class head on ImageNet MobileNetV3-Large features; no previous MangoLeaf weights, temperature, or predictions are loaded. Reuse V5's recorded **2,649 mango memberships**, **2,511 OE identities**, and **995 reporting identities** only after matching current files. Add the separately inventoried **71 mango photos**. The old 500 OOD photos are now OE training data, never reporting data.

The recipe follows V4.5.0: five head-training epochs at 0.001, ten fine-tuning epochs at 0.00001 with BatchNormalization frozen; batch size 32; weighted mango cross-entropy plus 0.5 uniform-target OE loss. Six augmentation categories and the same seven original validation preview identities are retained. Targets remain 70/15/15; historical split membership and whole related groups take priority over exact percentages.

**Reporting rule fixed before training:** accept maximum calibrated confidence >=70%; otherwise uncertain. No margin or entropy rejection thresholds or threshold search. Temperature is newly fitted on mango validation photos only. The evaluation notebook reports both highest-scoring predictions before rejection ("raw") and the same predictions with this fixed rule.

**Manual preparation:** upload the three supplied `reference-inputs` JSON files to `MyDrive/mangoleaf_v5_1_0_inputs/reference-inputs`. Make all five photo folders accessible in this account. Set their paths in Step 3, including the 995-photo collection's original 1,000-candidate folder. Choose a Colab GPU and Python 3.11/3.12. Run cells in order; restart the session after Step 1, rerun Step 1, then continue. Complete review cells only after inspecting their outputs. Do not rerun run creation to get past a review pause.

The historical runtime pins are candidate settings, not a promise of present compatibility. Fresh dependency audits and architecture conversion run before fitting and stop on new unresolved findings. The converter preserves narrowly scoped known packaging-operation advisories visibly. Save and inspect their reports. Training/results are not measured until these notebooks run successfully in Colab. Existing V4.5.0 application assets stay unchanged.

## Step 1: Mount Drive and install the candidate runtime

**What:** Prepare one ordinary Colab GPU session.

**Why:** Mixed pre-install and post-install modules invalidate a run.

**How:** Install the pinned research packages with binary wheels; restart, then rerun this cell.

**Expected result:** An installation receipt and logs. No photos or trained models are created.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

# Check Python before mounting Drive or installing scientific packages.
print('Notebook Python:', sys.version.split()[0])
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError(
        'Unsupported Python ' + sys.version.split()[0] + '. '
        'Open Runtime > Change runtime type, keep a GPU such as T4, and select '
        'Runtime Version 2026.04 (Python 3.12.13), then rerun Step 1. '
        'If unavailable, use a listed Python 3.11/3.12 runtime: '
        'https://research.google.com/colaboratory/runtime-version-faq.html. '
        'These TensorFlow 2.19.0 pins must not run on an unsupported Python version.'
    )

from google.colab import drive

drive.mount('/content/drive')

TRAINING_PACKAGES = [
    'tensorflow==2.19.0',
    'keras==3.15.1',
    'numpy==2.1.3',
    'Pillow==13.0.0',
    'ImageHash==4.3.2',
    'scipy==1.16.3',
    'matplotlib==3.11.2',
    'scikit-learn==1.9.1',
    'pip-audit==2.10.1',
    'packaging==25.0',
    'uv==0.13.4',
    # Explicit security pins prevent pip from retaining vulnerable provider helpers.
    'pip==26.2.1',
    'setuptools==84.1.0',
    'certifi==2026.10.18',
    'filelock==4.2.1',
    'idna==3.15',
    'msgpack==1.2.3',
    'requests==2.33.0',
    'urllib3==2.8.0',
    'werkzeug==3.1.9',
]
setup_directory = Path('/content/mangoleaf-v510-setup-20261101')
setup_directory.mkdir(exist_ok=True)
setup_receipt = setup_directory / 'installation.json'
installed = json.loads(setup_receipt.read_text()) if setup_receipt.is_file() else None

if installed is None:
    # Upgrade the installer first; this subprocess does not import scientific modules.
    pip_pin = next(pin for pin in TRAINING_PACKAGES if pin.startswith('pip=='))
    bootstrap = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
         '--only-binary=:all:', '--no-deps', pip_pin],
        capture_output=True, text=True, check=False,
    )
    (setup_directory / 'pip-bootstrap.log').write_text(bootstrap.stdout + '\n' + bootstrap.stderr)
    if bootstrap.returncode != 0:
        raise RuntimeError('Installer update failed. Read: ' + str(setup_directory / 'pip-bootstrap.log'))
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
         '--only-binary=:all:', *TRAINING_PACKAGES],
        capture_output=True, text=True, check=False,
    )
    (setup_directory / 'installation.log').write_text(result.stdout + '\n' + result.stderr)
    if result.returncode != 0:
        raise RuntimeError('Package installation failed. Read: ' + str(setup_directory / 'installation.log'))
    setup_receipt.write_text(json.dumps({
        'packages': TRAINING_PACKAGES, 'installationPid': os.getpid(),
        'python': sys.version, 'restartRequired': True,
    }, indent=2) + '\n')
    print('Installation complete. Restart session, then rerun Step 1 and continue.')
else:
    if installed['packages'] != TRAINING_PACKAGES:
        raise RuntimeError('Setup pins changed. Use a fresh runtime rather than mixing installation attempts.')
    print('Packages already installed. Restart completed:', installed['installationPid'] != os.getpid())

## Step 2: Audit dependencies and verify GPU execution

**What:** Check imported versions and the complete research dependency closure.

**Why:** An installation success alone does not establish runtime compatibility or security.

**How:** Run a fresh pip-audit, retain provider conflicts, and execute a real GPU matrix multiplication.

**Expected result:** Version, audit, and GPU records, or a blocking error with its log.

In [ ]:
import csv
import hashlib
import importlib.metadata
import math
import random
import shutil
import uuid
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import PurePosixPath

# Stop before scientific imports can mix pre-install modules with new files.
installed = json.loads(setup_receipt.read_text())
if installed['installationPid'] == os.getpid():
    raise RuntimeError('Restart the Colab session after installation, rerun Step 1, then rerun this cell.')

import imagehash
import matplotlib.pyplot as plt
import numpy as np
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
from PIL import Image, ImageDraw, ImageOps
from scipy.optimize import minimize_scalar
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

for pin in TRAINING_PACKAGES:
    name, expected = pin.split('==')
    if importlib.metadata.version(name) != expected:
        raise RuntimeError('Runtime version differs from chosen pin: ' + pin)
if tf.__version__ != '2.19.0' or tf.keras.__version__ != '3.15.1':
    raise RuntimeError('Imported TensorFlow/Keras versions differ from installed pins; restart a clean session.')

# Audit only the transitive research runtime, rather than treating every
# unrelated preinstalled Colab provider package as an executed experiment dependency.
def installed_research_dependencies(pins):
    """Collect the actual transitive research dependency closure."""
    names = set()
    examined_contexts = set()
    pending_requirements = [Requirement(pin) for pin in pins]
    while pending_requirements:
        requirement = pending_requirements.pop()
        name = canonicalize_name(requirement.name)
        distribution = importlib.metadata.distribution(name)
        if not requirement.specifier.contains(distribution.version, prereleases=True):
            raise RuntimeError(
                'Research dependency conflict: ' + str(requirement)
                + '; installed ' + distribution.version
            )
        names.add(name)
        # Extras such as CacheControl[filecache] add dependencies that must be audited.
        # Revisit a package when a later requirement enables a different extra.
        for extra in {''} | requirement.extras:
            context = (name, extra)
            if context in examined_contexts:
                continue
            examined_contexts.add(context)
            for dependency in distribution.requires or []:
                child = Requirement(dependency)
                if child.marker is None or child.marker.evaluate({'extra': extra}):
                    pending_requirements.append(child)
    return names


research_names = installed_research_dependencies(TRAINING_PACKAGES)

runtime_lock = setup_directory / 'training-runtime-lock.txt'
runtime_lock.write_text(''.join(
    name + '==' + importlib.metadata.version(name) + '\n'
    for name in sorted(research_names)
), encoding='utf-8')
audit_path = setup_directory / 'training-audit.json'
audit = subprocess.run([
    sys.executable, '-m', 'pip_audit', '-r', str(runtime_lock),
    '--no-deps', '--disable-pip', '--format=json', '--output', str(audit_path),
    '--progress-spinner=off',
], capture_output=True, text=True, check=False)
(setup_directory / 'training-audit.log').write_text(audit.stdout + '\n' + audit.stderr)
audit_report = json.loads(audit_path.read_text()) if audit_path.is_file() else {}
dependencies = audit_report.get('dependencies', [])
findings = [row for row in dependencies if row.get('vulns')]
skipped = [row for row in dependencies if row.get('skip_reason')]
audited_names = {canonicalize_name(row['name']) for row in dependencies if row.get('name')}
if audit.returncode != 0 or findings or skipped or audited_names != research_names:
    print('Dependency audit exit:', audit.returncode)
    for row in findings:
        print('Unresolved package:', row['name'], row.get('version', 'unknown'))
        details = sorted({
            vulnerability.get('id', 'unknown') + ' (fix versions: '
            + ', '.join(vulnerability.get('fix_versions', [])) + ')'
            for vulnerability in row['vulns']
        })
        for detail in details:
            print('  ', detail)
    for row in skipped:
        print('Skipped package:', row.get('name', 'unknown'), row['skip_reason'])
    print('Missing packages:', sorted(research_names - audited_names))
    print('Unexpected packages:', sorted(audited_names - research_names))
    print('Full audit and log:', audit_path, setup_directory / 'training-audit.log')
    raise RuntimeError('Training dependency audit incomplete or unresolved. Read ' + str(audit_path))

provider_check = subprocess.run(
    [sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True, check=False,
)
(setup_directory / 'provider-pip-check.log').write_text(provider_check.stdout + '\n' + provider_check.stderr)
if provider_check.returncode:
    print('Colab provider compatibility findings remain visible:', setup_directory / 'provider-pip-check.log')

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('Select a Colab GPU runtime before training.')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
with tf.device('/GPU:0'):
    gpu_probe = tf.linalg.matmul(tf.ones((16, 16)), tf.ones((16, 16)))
if 'GPU' not in gpu_probe.device or not np.isfinite(gpu_probe.numpy()).all():
    raise RuntimeError('The GPU execution probe failed.')
RUNTIME = {
    'python': sys.version, 'tensorflow': tf.__version__,
    'packages': {name: importlib.metadata.version(name) for name in sorted(research_names)},
    'trainingDependencyAuditPassed': True, 'providerPipCheckReturnCode': provider_check.returncode,
    'gpuCount': len(gpus), 'gpuProbeDevice': gpu_probe.device,
    'execution': 'ordinary Colab Python cells; no custom cell magic',
    'browserParityPassed': False,
}
print('TensorFlow:', tf.__version__, 'Keras:', tf.keras.__version__, 'GPU count:', len(gpus))

## Step 3: Set input paths and create this account's output run

**What:** Locate photos and hash-pinned reference records.

**Why:** Input lists describe past identities; they do not authorize loading old MangoLeaf weights.

**How:** Edit the six input paths if needed. Verify reference bytes and isolate the new output directory.

**Expected result:** A unique V5.1.0 run under this account's mangoleaf_experiments folder.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
RAW_MANGO_INPUT = DRIVE_ROOT / 'final_mangoleafai_dataset_public_and_bpi'
NEW_MANGO_INPUT = DRIVE_ROOT / 'new_dataset'
OE_INPUT = DRIVE_ROOT / 'mangoleafai_oe_images'
OLD_OOD_INPUT = DRIVE_ROOT / 'mango_ood_test_v4_4_0'
OOD_INPUT = DRIVE_ROOT / 'fresh_ood_collections/open-images-cvdf-validation-1000-20261018T183623Z-ff6129df/images'
# Upload the supplied reference-inputs folder here, keeping its three filenames.
REFERENCE_ROOT = DRIVE_ROOT / 'mangoleaf_v5_1_0_inputs/reference-inputs'
OUTPUT_ROOT = DRIVE_ROOT / 'mangoleaf_experiments/v5.1.0/training'
ALLOW_SHARED_INPUT_SHORTCUTS = False  # Enable only for reviewed My Drive input shortcuts.

EXPERIMENT_VERSION = 'v5.1.0'
MODEL_VERSION = 'model-v5.1.0-candidate'
CLASS_NAMES = ['anthracnose', 'bacterial_black_spot', 'bacterial_canker',
               'healthy_fruit', 'healthy_leaf', 'mango_scab', 'sooty_mold']
SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
PHASH_DISTANCE = 5
STAGE1_EPOCHS = 5
STAGE2_EPOCHS = 10
STAGE1_LEARNING_RATE = 0.001
STAGE2_LEARNING_RATE = 0.00001
OE_LOSS_WEIGHT = 0.5
CONTRAST_FACTOR = 0.1
CONFIDENCE_THRESHOLD = 0.70  # Fixed before test predictions; no threshold search.
EXPECTED_OOD_IMAGE_COUNT = 995

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

def drive_input_path(path, allow_shared=False, my_drive=Path('/content/drive/MyDrive')):
    """Accept selected My Drive inputs, with explicit shared-shortcut permission."""
    selected = Path(path)
    my_drive = Path(my_drive)
    if (not selected.is_absolute() or '..' in selected.parts
            or selected == my_drive or not selected.is_relative_to(my_drive)):
        raise ValueError('Select an input folder through a path below My Drive: ' + str(selected))

    resolved = selected.resolve()
    owned_root = my_drive.resolve()
    shortcut_root = (my_drive.parent / '.shortcut-targets-by-id').resolve()
    owned = resolved != owned_root and resolved.is_relative_to(owned_root)
    shared = (resolved.is_relative_to(shortcut_root)
              and len(resolved.relative_to(shortcut_root).parts) >= 2)
    print('Input:', selected, '\nResolved:', resolved)
    if not owned and not (allow_shared and shared):
        raise ValueError(
            'Input resolves outside My Drive. For a reviewed shared input, add its shortcut '
            'to My Drive and set ALLOW_SHARED_INPUT_SHORTCUTS = True. '
            'Other external paths remain unsupported: ' + str(resolved)
        )
    if not resolved.is_dir():
        raise FileNotFoundError('Input folder missing or inaccessible in the mounted account: ' + str(selected))
    return selected


def owned_my_drive_path(path, my_drive=Path('/content/drive/MyDrive')):
    """Keep outputs and completed training runs in the mounted account's My Drive."""
    selected = Path(path)
    resolved = selected.resolve()
    owned_root = Path(my_drive).resolve()
    print('Selected location:', selected)
    print('Resolved location:', resolved)
    print('Exists:', resolved.exists())
    if resolved == owned_root or not resolved.is_relative_to(owned_root):
        raise ValueError(
            'Output and training-run paths must resolve inside this mounted account\'s '
            'My Drive. Select its actual mangoleaf_experiments folder, not a shortcut '
            'to another account\'s output folder: ' + str(resolved)
        )
    return resolved

REFERENCE_SHA256 = {'v5-training-input-manifest.json': '7d6bb3e80edd5c96731838c172deec7e836d3f877d8f8fa7007458df165db8cf', 'v5-evaluation-reporting-lock.json': 'b35725b99c960557aca9740665863c012d15053475ceae5efac6bd237fdbaab0', 'new-dataset-identities.json': 'dc45dd9424b6f5eee387025ce03b3a70e51ba5d9e8f41e3d8728ccdc4b3f915b'}


input_roots = (RAW_MANGO_INPUT, NEW_MANGO_INPUT, OE_INPUT, OLD_OOD_INPUT, OOD_INPUT, REFERENCE_ROOT)
for input_path in input_roots:
    drive_input_path(input_path, allow_shared=ALLOW_SHARED_INPUT_SHORTCUTS)
OUTPUT_ROOT = owned_my_drive_path(OUTPUT_ROOT)
resolved_inputs = [path.resolve() for path in input_roots]
for index, first in enumerate(resolved_inputs):
    for second in resolved_inputs[index + 1:] + [OUTPUT_ROOT]:
        if first == second or first.is_relative_to(second) or second.is_relative_to(first):
            raise ValueError('Input, reference, and output directories must be separate.')
reference_values = {}
for name, expected in REFERENCE_SHA256.items():
    data = (REFERENCE_ROOT / name).read_bytes()
    if hashlib.sha256(data).hexdigest() != expected:
        raise RuntimeError('Reference record differs from supplied bytes: ' + name)
    reference_values[name] = json.loads(data)
parent_manifest = reference_values['v5-training-input-manifest.json']
ood_reference = reference_values['v5-evaluation-reporting-lock.json']['images']
additional_reference = reference_values['new-dataset-identities.json']['images']
if (len(parent_manifest['mango']), len(parent_manifest['oe']), len(ood_reference), len(additional_reference)) != (2649, 2511, 995, 71):
    raise RuntimeError('Reference counts differ from the approved experimental design.')

# Run this cell once per attempt. Review pauses resume at their own cell.
run_id = 'v5-1-0-training-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
DATASET_DIR = RUN_DIR / 'dataset'
FIGURE_DIR = RUN_DIR / 'figures'
FIGURE_DIR.mkdir()
shutil.copytree(setup_directory, RUN_DIR / 'runtime_preflight')
(RUN_DIR / 'reference-inputs').mkdir()
for name in REFERENCE_SHA256:
    shutil.copyfile(REFERENCE_ROOT / name, RUN_DIR / 'reference-inputs' / name)
(RUN_DIR / 'runtime.json').write_text(json.dumps(RUNTIME, indent=2) + '\n')
print('Preserve this exact training run:', RUN_DIR)

## Step 4: Verify the recorded photos and inventory the additions

**What:** Match all selected files and decoded pixels to the saved records.

**Why:** Raw folders contain extra files and the reporting folder originally contains 1,000 candidates.

**How:** Hash files, select only recorded identities, decode them, and inventory the 71 new class-labeled photos. Completed pool inventories are checkpointed; rerunning this cell reuses them when metadata/runtime match. Exact bytes are verified again before snapshots/training. Derived pHash differences are audited separately. Originals remain untouched.

**Expected result:** 2,649 original mango, 71 added mango, 2,511 unique OE, and 995 reporting photos, or an explicit mismatch.

In [ ]:
import time
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}
PREPROCESSING = 'RGB decode; bilinear 224x224 without antialias; float32 0-255; MobileNetV3 internal rescaling'

def file_sha256(path):
    """Hash file bytes without loading the whole image into memory."""
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def object_sha256(value):
    """Identify structured configuration with explicit canonical JSON."""
    serialized = json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False)
    return hashlib.sha256(serialized.encode('utf-8')).hexdigest()


def read_json(path):
    """Read recorded UTF-8 JSON without altering its bytes."""
    return json.loads(Path(path).read_text(encoding='utf-8'))


def save_json_once(path, value):
    """Write evidence once and reject a changed overwrite."""
    path = Path(path)
    if path.exists():
        if read_json(path) != value:
            raise RuntimeError('Refusing to overwrite changed evidence: ' + str(path))
        return
    with path.open('x', encoding='utf-8', newline='\n') as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write('\n')


def safe_child(root, relative):
    """Resolve a validated relative path inside its declared root."""
    if not isinstance(relative, str) or not relative or '\\' in relative or ':' in relative or '\x00' in relative:
        raise ValueError('Unsafe relative path: ' + str(relative))
    child = PurePosixPath(relative)
    if child.is_absolute() or '..' in child.parts or child == PurePosixPath('.'):
        raise ValueError('Unsafe relative path: ' + relative)
    destination = Path(root).joinpath(*child.parts).resolve()
    destination.relative_to(Path(root).resolve())
    return destination


def raw_rgb(path):
    """Decode and orient original image pixels for identity checks."""
    with Image.open(path) as opened:
        return ImageOps.exif_transpose(opened).convert('RGB')


def load_pixels(path):
    # TensorFlow matches the app's bilinear float32 input. Pillow handles WebP
    # content, including WebP files whose suffix claims JPEG.
    """Match the recorded 224x224 bilinear model preprocessing."""
    with Path(path).open('rb') as handle:
        signature = handle.read(12)
    if signature[:4] == b'RIFF' and signature[8:12] == b'WEBP':
        image = tf.convert_to_tensor(np.asarray(raw_rgb(path)), dtype=tf.uint8)
    else:
        image = tf.io.decode_image(tf.io.read_file(str(path)), channels=3, expand_animations=False)
    return tf.image.resize(tf.cast(image, tf.float32), IMAGE_SIZE, method='bilinear', antialias=False)


def portable_rows(rows):
    """Remove machine-local paths from portable image records."""
    return [{key: value for key, value in row.items() if key != 'path'} for row in rows]


def verify_image_rows(rows):
    """Detect changed photo bytes before fitting or reporting."""
    for row in rows:
        if file_sha256(row['path']) != row['sha256']:
            raise RuntimeError('Input image changed after inventory: ' + row['path'])


def image_paths(folder):
    """List supported image files in deterministic path order."""
    return sorted(
        path for path in Path(folder).rglob('*')
        if path.suffix.lower() in IMAGE_EXTENSIONS and path.is_file()
    )


def image_inventory(folder, pool, labelled=False, paths=None):
    """Measure image identities, dimensions, and decode validity."""
    rows = []
    failures = []
    paths = image_paths(folder) if paths is None else paths
    started = time.monotonic()
    print(pool, 'inventory started:', len(paths), 'images', flush=True)
    for index, path in enumerate(paths, start=1):
        relative = path.relative_to(folder).as_posix()
        try:
            pixels = raw_rgb(path)
            loaded = load_pixels(path).numpy()
            if loaded.shape != (224, 224, 3) or not np.isfinite(loaded).all():
                raise ValueError('Invalid decoded model input')
            sha = file_sha256(path)
            decoded = hashlib.sha256(str(pixels.size).encode('utf-8') + pixels.tobytes()).hexdigest()
            row = {
                'id': sha[:20], 'pool': pool, 'path': str(path),
                'sourceRelativePath': relative, 'sha256': sha,
                'decodedRgbSha256': decoded, 'bytes': path.stat().st_size,
                'phash': str(imagehash.phash(pixels, hash_size=8)),
                'dimensions': list(pixels.size),
            }
            if labelled:
                row['label'] = PurePosixPath(relative).parts[0]
                if row['label'] not in CLASS_NAMES:
                    raise ValueError('Unexpected class folder: ' + row['label'])
            rows.append(row)
        except Exception as error:
            failures.append({'relativePath': relative, 'error': str(error)})
        if index % 100 == 0 or index == len(paths):
            print(pool, 'images processed:', index, '/', len(paths),
                  'elapsed seconds:', round(time.monotonic() - started, 1), flush=True)
    if failures:
        save_json_once(RUN_DIR / (pool + '_inventory_failures.json'), failures)
        raise RuntimeError('Correct unreadable or mislabeled files. First failures: ' + json.dumps(failures[:10]))
    identities = defaultdict(set)
    for row in rows:
        identities[row['id']].add(row['sha256'])
    if any(len(values) != 1 for values in identities.values()):
        raise RuntimeError('Truncated image ID collision. Use full SHA-256 IDs for a new run.')
    return rows


def save_contact_sheets(rows, destination, title, columns=4):
    """Render numbered originals for manual inspection."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    page_count = math.ceil(len(rows) / 20)
    print(destination.name, 'preview sheets started:', page_count, 'pages', flush=True)
    for page, start in enumerate(range(0, len(rows), 20), start=1):
        chosen = rows[start:start + 20]
        sheet = Image.new('RGB', (columns * 280, math.ceil(len(chosen) / columns) * 295 + 40), 'white')
        drawing = ImageDraw.Draw(sheet)
        drawing.text((8, 8), title, fill='black')
        for index, row in enumerate(chosen):
            picture = raw_rgb(row['path'])
            picture.thumbnail((268, 240))
            left = (index % columns) * 280 + 6
            top = (index // columns) * 295 + 40
            sheet.paste(picture, (left, top))
            drawing.text((left, top + 242), f"Photo {start + index + 1}: {row['id']}\n{row['sourceRelativePath'][:38]}", fill='black')
        sheet.save(destination / f'page_{page:03d}.png')
        if page % 10 == 0 or page == page_count:
            print(destination.name, 'preview pages saved:', page, '/', page_count, flush=True)



def reconcile_recorded_images(root, expected_rows, observed_rows, pool, labelled):
    """Require exact identities while auditing, rather than equating, derived pHash."""
    if len(expected_rows) != len(observed_rows):
        raise RuntimeError('Incomplete decoded inventory: ' + pool)
    result, drift = [], []
    for expected, observed in zip(expected_rows, observed_rows):
        relative = observed.get('currentSourceRelativePath', observed['sourceRelativePath'])
        path = safe_child(root, relative)
        if str(path) != str(Path(observed['path']).resolve()):
            raise RuntimeError('Cached image path differs from its input root.')
        for key in ('sha256', 'decodedRgbSha256', 'bytes', 'dimensions'):
            if key not in observed or key in expected and observed[key] != expected[key]:
                raise RuntimeError('Recorded photo content or decode differs: ' + relative + ' / ' + key)
        if labelled and (observed.get('label') != expected['label'] or PurePosixPath(relative).parts[0] != expected['label']):
            raise RuntimeError('Recorded photo is outside its expected class: ' + relative)
        current_phash = observed.get('phash', '')
        if not isinstance(current_phash, str) or len(current_phash) != 16 or any(c not in '0123456789abcdef' for c in current_phash):
            raise RuntimeError('Malformed current 64-bit perceptual hash: ' + relative)
        merged = dict(expected)
        for key, value in observed.items():
            if key not in merged:
                merged[key] = value
        merged.update(path=str(path), pool=pool, currentSourceRelativePath=relative, phash=current_phash)
        if 'phash' in expected:
            historical = expected['phash']
            merged['referencePhash'] = historical
            if historical != current_phash:
                drift.append({'id': expected.get('id', observed['id']), 'sha256': observed['sha256'],
                    'sourceRelativePath': relative, 'referencePhash': historical, 'currentPhash': current_phash,
                    'distance': (int(historical, 16) ^ int(current_phash, 16)).bit_count(),
                    'exactFileAndDecodedPixelsMatch': True,
                    'cause': 'Derived fingerprint differs; the specific library/platform cause is not established.'})
        result.append(merged)
    return result, drift


def inventory_checkpoint_identity(root, expected_rows, pool, labelled):
    """Bind cached decoding to this pool, reference list, runtime, and root."""
    return {'schemaVersion': 2, 'root': str(Path(root).resolve()), 'pool': pool, 'labelled': labelled,
            'expectedRowsSha256': object_sha256(expected_rows), 'runtimeSha256': object_sha256(RUNTIME),
            'phashAlgorithm': 'ImageHash phash RGB input; hash_size=8; current runtime'}


def write_inventory_checkpoint(root, expected_rows, pool, labelled, rows, discovered_count):
    """Save a completed decoding pass without overwriting older evidence."""
    reconcile_recorded_images(root, expected_rows, rows, pool, labelled)
    identity = inventory_checkpoint_identity(root, expected_rows, pool, labelled)
    directory = RUN_DIR / 'inventory_cache_v2'
    directory.mkdir(exist_ok=True)
    entries = []
    for row in rows:
        path = Path(row['path'])
        stat = path.stat()
        entries.append({**row, 'checkpointFileSize': stat.st_size, 'checkpointMtimeNs': stat.st_mtime_ns})
    save_json_once(directory / (object_sha256(identity) + '.json'), {
        'identity': identity, 'rows': entries, 'discoveredFileCount': discovered_count,
        'rowsSha256': object_sha256(entries),
        'reuseLimit': 'Only decoded inventory work is reused. Exact bytes are verified before snapshots and training.',
    })


def read_inventory_checkpoint(root, expected_rows, pool, labelled):
    """Reuse unchanged decoding records; changed input files still fail closed."""
    identity = inventory_checkpoint_identity(root, expected_rows, pool, labelled)
    checkpoint = RUN_DIR / 'inventory_cache_v2' / (object_sha256(identity) + '.json')
    recovered = globals().get('_V510_STEP4_RECOVERED', {}).get(pool)
    if not checkpoint.exists() and recovered is not None:
        if recovered['root'] != str(Path(root).resolve()) or recovered['expectedRowsSha256'] != object_sha256(expected_rows):
            raise RuntimeError('Recovered inventory belongs to different inputs: ' + pool)
        write_inventory_checkpoint(root, expected_rows, pool, labelled, recovered['rows'], recovered['discoveredFileCount'])
    if not checkpoint.exists():
        return None
    saved = read_json(checkpoint)
    if saved['identity'] != identity or object_sha256(saved['rows']) != saved['rowsSha256']:
        raise RuntimeError('Inventory checkpoint differs from its recorded identity: ' + pool)
    reconcile_recorded_images(root, expected_rows, saved['rows'], pool, labelled)
    for row in saved['rows']:
        stat = Path(row['path']).stat()
        if stat.st_size != row['checkpointFileSize'] or stat.st_mtime_ns != row['checkpointMtimeNs']:
            raise RuntimeError('A checkpointed input changed. Preserve this attempt and inspect: ' + row['path'])
    # Cache bookkeeping must not enter scientific image manifests.
    return {**saved, 'rows': [{key: value for key, value in row.items()
                             if key not in ('checkpointFileSize', 'checkpointMtimeNs')} for row in saved['rows']]}


def select_recorded_images(root, expected_rows, pool, labelled=False):
    """Locate only recorded identities; extra raw-folder files never enter training."""
    root = Path(root)
    if len({row['sha256'] for row in expected_rows}) != len(expected_rows):
        raise ValueError('Recorded membership contains repeated file identities: ' + pool)
    cached = read_inventory_checkpoint(root, expected_rows, pool, labelled)
    if cached is not None:
        current = cached['rows']
        discovered_count = cached['discoveredFileCount']
        print(pool, 'reusing completed inventory:', len(current), 'photos; bytes checked again before snapshot/training.', flush=True)
    else:
        paths = image_paths(root)
        by_hash = defaultdict(list)
        for index, path in enumerate(paths, 1):
            safe_child(root, path.relative_to(root).as_posix())
            by_hash[file_sha256(path)].append(path)
            if index % 500 == 0:
                print(pool, 'file hashes checked:', index, '/', len(paths), flush=True)
        selected = []
        for expected in expected_rows:
            matches = by_hash.get(expected['sha256'], [])
            if labelled:
                matches = [path for path in matches if path.relative_to(root).parts[0] == expected['label']]
            if not matches:
                raise RuntimeError('Recorded photo missing, changed, or outside its class folder: ' + str(expected.get('sourceRelativePath', expected.get('relativePath'))))
            relative = expected.get('sourceRelativePath', expected.get('relativePath', ''))
            chosen = min(matches, key=lambda path: (path.relative_to(root).as_posix() != relative, path.as_posix()))
            selected.append(chosen)
        current = image_inventory(root, pool, labelled=labelled, paths=selected)
        discovered_count = len(paths)
        # Checkpoint decoding before comparing derived fingerprints, so a later
        # issue does not discard a completed expensive inventory.
        write_inventory_checkpoint(root, expected_rows, pool, labelled, current, discovered_count)
    result, drift = reconcile_recorded_images(root, expected_rows, current, pool, labelled)
    save_json_once(RUN_DIR / (pool + '_input_selection_v2.json'), {
        'selectedCount': len(result), 'discoveredFileCount': discovered_count,
        'rows': portable_rows(result), 'selection': 'recorded SHA-256 and decoded-pixel identities',
        'derivedPhashDrift': drift,
        'phashPolicy': 'Use current-runtime phash consistently; preserve historical values separately.',
    })
    if drift:
        print(pool, 'unchanged photos with different similarity fingerprints:', len(drift), flush=True)
    return result


raw_rows = select_recorded_images(RAW_MANGO_INPUT, parent_manifest['mango'], 'original_mango', labelled=True)
new_paths = image_paths(NEW_MANGO_INPUT)
if len(new_paths) != 71:
    raise RuntimeError('Expected exactly 71 additional image files, not folders; found ' + str(len(new_paths)))
new_rows = select_recorded_images(NEW_MANGO_INPUT, additional_reference, 'new_mango', labelled=True)
original_oe_rows = select_recorded_images(OE_INPUT, [r for r in parent_manifest['oe'] if r['pool'] == 'original_oe'], 'original_oe')
transferred_rows = select_recorded_images(OLD_OOD_INPUT, [r for r in parent_manifest['oe'] if r['pool'] == 'transferred_ood'], 'transferred_ood')
approved_oe_rows = original_oe_rows + transferred_rows
held_out_ood_rows = select_recorded_images(OOD_INPUT, ood_reference, 'reporting_ood')
save_json_once(RUN_DIR / 'phash_drift_audit.json', {
    'pools': {pool: read_json(RUN_DIR / (pool + '_input_selection_v2.json'))['derivedPhashDrift']
              for pool in ('original_mango', 'new_mango', 'original_oe', 'transferred_ood', 'reporting_ood')},
    'runtime': RUNTIME, 'policy': 'All comparisons use current-runtime phash; historical values remain unchanged in references.',
})
for name, rows in [('mango_inventory.json', raw_rows), ('new_mango_inventory.json', new_rows),
                   ('oe_inventory.json', approved_oe_rows), ('held_out_ood_inventory.json', held_out_ood_rows)]:
    save_json_once(RUN_DIR / name, {'images': portable_rows(rows)})
save_contact_sheets(new_rows, RUN_DIR / 'new_image_previews', 'V5.1.0 additional mango photos: inspect labels and related captures')
print('Selected original mango:', len(raw_rows), 'additional:', len(new_rows),
      'OE:', len(approved_oe_rows), 'reporting OOD:', len(held_out_ood_rows))
print('The five previously excluded OOD candidates stay excluded; no model predictions ran.')

## Step 5: Check the new-photo relationships

**What:** Preserve historical reviewed identities and inspect new duplicate/similarity candidates.

**Why:** Changed inputs still need overlap checks; historical labels and source limitations remain visible.

**How:** Reject exact cross-pool leakage; compare new photos against mango, OE, and reporting photos. Create numbered inspection sheets.

**Expected result:** Read-only audit and preview files; no automatic deletion or expert-verification claim.

In [ ]:
REVIEWED_IMAGES = [
    {
        "label": "anthracnose",
        "currentName": "verified mango leaf disease 21.jpg",
        "historicalName": "verified mango leaf disease 21.jpg",
        "sha256": "a557157b1e887d6f8f7067efbaa97447074114b2967c3c2337656d1936b45102"
    },
    {
        "label": "anthracnose",
        "currentName": "verified mango leaf disease 20.jpg",
        "historicalName": "verified mango leaf disease 20.jpg",
        "sha256": "87dad59aa6422d959e3fcfffd35ddca9c918e858dd7b220f1e016ae09d741f6f"
    },
    {
        "label": "anthracnose",
        "currentName": "DSC_0119.JPG",
        "historicalName": "DSC_0119.JPG",
        "sha256": "01e6fd123c9ae054d2d904fef25f543279c6c74914d44a33cd9c7488ca2861c6"
    },
    {
        "label": "anthracnose",
        "currentName": "DSC_0488.JPG",
        "historicalName": "DSC_0488.JPG",
        "sha256": "ee9676472204c9cdd0830dce3be63a9d47e646ecb9d2c70126ddcdac063e4ecd"
    },
    {
        "label": "anthracnose",
        "currentName": "DSC_0970.JPG",
        "historicalName": "DSC_0970.JPG",
        "sha256": "8b5d0b0e2500ff762a773b326c6a0a3ab545e51e682a253d3776be738d61123c"
    },
    {
        "label": "anthracnose",
        "currentName": "Anthracnose on young leaves.jpeg",
        "historicalName": "Anthracnose on young leaves.jpeg",
        "sha256": "5c10b4851fd67db6b5a21677e725d124491633afea45186fbe2e2a6f9bd554ec"
    },
    {
        "label": "anthracnose",
        "currentName": "AnthracnoseonMangoLeaf.jpeg",
        "historicalName": "AnthracnoseonMangoLeaf.jpeg",
        "sha256": "52a0fa64a504139aa65e6cbf807ecae21be6f4933bc4c1d5e2d8797a54de7227"
    },
    {
        "label": "sooty_mold",
        "currentName": "IMG_20260219_135000.jpg",
        "historicalName": "IMG_20260219_135000.jpg",
        "sha256": "2a22ee7f89ea03ad9ee924b5815ae71576d6ccd3305576dccc6f3bf00e31fce3"
    },
    {
        "label": "sooty_mold",
        "currentName": "viber_image_2024-11-15_13-17-51-567.jpg",
        "historicalName": "viber_image_2024-11-15_13-17-51-567.jpg",
        "sha256": "d785e72ca8291a194d5f56f2c75d2087ca3fd9728549246b888a3b964ce5b3c9"
    },
    {
        "label": "sooty_mold",
        "currentName": "IMG_20260219_134935.jpg",
        "historicalName": "IMG_20260219_134935.jpg",
        "sha256": "6c1c6c90f2dbfdb3ab1d451b90a9ce2769222aaa237bb820bf55f782e63498d5"
    },
    {
        "label": "sooty_mold",
        "currentName": "DSC_1029.JPG",
        "historicalName": "DSC_1029.JPG",
        "sha256": "7dfd201d72ea78a4557fdc5c5bddb594af538ff0270e020e73c9f382b3fa6f74"
    },
    {
        "label": "sooty_mold",
        "currentName": "724869371_27534362222843583_7670640125862653147_n.jpg",
        "historicalName": "724869371_27534362222843583_7670640125862653147_n.jpg",
        "sha256": "81c5a02299de1f2d66627a0f709a4f0c3235066749f7f8b203840b86f8a93b3c"
    },
    {
        "label": "sooty_mold",
        "currentName": "45a36510-0844-4baa-9795-3de7e0236896.jpg",
        "historicalName": "45a36510-0844-4baa-9795-3de7e0236896.jpg",
        "sha256": "32d90fe257d7ef563ef8166768b638a92a95c8196d33de90ced349c22b62a410"
    },
    {
        "label": "healthy_leaf",
        "currentName": "HealthyLeaf_2.jpg",
        "historicalName": "HealthyLeaf_2.jpg",
        "sha256": "e2e68e23f0fd8f88b999c0740cb577e126e78de61fa1fe71772d908ea2fc6375"
    },
    {
        "label": "healthy_leaf",
        "currentName": "HealthyLeaf_1.jpg",
        "historicalName": "HealthyLeaf_1.jpg",
        "sha256": "90a7907f7a259e404f6bb8c6558bb1081f2d2942249ebb5cac4b747a2f989e0c"
    },
    {
        "label": "mango_scab",
        "currentName": "15289a55-e368-4f61-a25c-85b2468243fd.jpg",
        "historicalName": "15289a55-e368-4f61-a25c-85b2468243fd.jpg",
        "sha256": "5a10c88561c498889b7668319c36fa1a432b942c21832e65d8c4a3063dbcf78b"
    },
    {
        "label": "mango_scab",
        "currentName": "DSC_0281.JPG",
        "historicalName": "DSC_0281.JPG",
        "sha256": "da1e429fc670c52fd75ba8cfe37b019606841ee7e012182f5696895e1ba13132"
    },
    {
        "label": "mango_scab",
        "currentName": "DSC_0966.JPG",
        "historicalName": "DSC_0966.JPG",
        "sha256": "de8cd5a96aefcccbd838a67411d44f13dcf9bf647200e40c71b7da28f26a1724"
    },
    {
        "label": "mango_scab",
        "currentName": "Scab.JPG",
        "historicalName": "Scab.JPG",
        "sha256": "10344e2e95ceeafab7d77e3e28e4ebbdebd699ec7e862c9c011516038e821068"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 70.jpg",
        "historicalName": "non verified mango disease 70.jpg",
        "sha256": "47a8192ddf400d3ba6397ba7d3acb21142d75267b80abe3911ba0d76e85ce995"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 71.jpg",
        "historicalName": "non verified mango disease 71.jpg",
        "sha256": "0c46b7f3c43190fb6e5663d6248f81e1eca31287ea8ab8cf3c2c9113aa46a848"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 73.jpg",
        "historicalName": "non verified mango disease 73.jpg",
        "sha256": "7562eee067e4a968c8f7c1f3281125bcb3546dcab2795f196ee11286e836ac09"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 68.jpg",
        "historicalName": "non verified mango disease 68.jpg",
        "sha256": "281064014381e3f9d4cd163b62ec6e643364ee5a3a0840875f9c310912a30c8d"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 67.jpg",
        "historicalName": "non verified mango disease 67.jpg",
        "sha256": "6e99067feab48b78d0e0c4260e342c1b64673dab00a5c18099310bb9053f1dd7"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 66.jpg",
        "historicalName": "non verified mango disease 66.jpg",
        "sha256": "0e515243d9cec0d0e1f970d4ba79081606c2280ee02176e30fa24b2817784cda"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 65.jpg",
        "historicalName": "non verified mango disease 65.jpg",
        "sha256": "ae4bfcc5cd3eba0f652aa4355d1780ef6efe27af71f3d020a15b1eee2c0f975b"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 56.jpg",
        "historicalName": "non verified mango disease 56.jpg",
        "sha256": "ad4415243dc86b38c7ac7f529fdb2e09f7fc9410ded477a553a0f33ef02736fa"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 53.jpg",
        "historicalName": "non verified mango disease 53.jpg",
        "sha256": "89652a16e8c0590a4fd51b9ff66c7ff9fb95a7ded8aece63c85565dad39f5fff"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 52.jpg",
        "historicalName": "non verified mango disease 52.jpg",
        "sha256": "cfb52caae3133082bc5f70c82f4b4c73589a65087409383e87453d405648391e"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 51.jpg",
        "historicalName": "non verified mango disease 51.jpg",
        "sha256": "e2c7e67b3026efe9eac0941272a41654f54c8317a2d11d7ba0eb81699891a2bd"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 49.jpg",
        "historicalName": "non verified mango disease 49.jpg",
        "sha256": "7b87cb61eb26ffffde585b8cbb254795f0da834ce1c306bae76fe2b6cdf0f736"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 11.jpg",
        "historicalName": "non verified mango disease 11.jpg",
        "sha256": "81a2f9abde6762ed891db43c570a5339d465682e3817c4f79917f8f497ed77cd"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 10.jpg",
        "historicalName": "verified mango disease 10.jpg",
        "sha256": "afcb278a84a3abf145a2d396bbcfa99b5818b217764c3fef07246d6993421095"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 9.jpg",
        "historicalName": "verified mango disease 9.jpg",
        "sha256": "6fdc1ff1319c85198ae2d42978e9c3ec11ee82b754e982915a3df896e922e187"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 1.jpg",
        "historicalName": "verified mango disease 1.jpg",
        "sha256": "156d089cee4c22e7256a0aa6ef48c84025507fcafda3b463fe39f9d9f852cd87"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 58.jpg",
        "historicalName": "verified mango disease 58.jpg",
        "sha256": "7fe031c22293d0c980ddf2744d2b3732bcbe22c659281c67ef7f24ce577619c6"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 36.jpg",
        "historicalName": "verified mango disease 36.jpg",
        "sha256": "22deb301d133db0af636c788508379203b6b90e4ca3ce9aa3ce238069cccfc4e"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 32.jpg",
        "historicalName": "verified mango disease 32.jpg",
        "sha256": "bbfbb925ba9bc8cb2595d501ab8180bc7378dfb199a1110793743e79a40e376f"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 6.jpg",
        "historicalName": "verified mango disease 6.jpg",
        "sha256": "262ea4d6d82594a4333470180b3de1a77560fafc3acc1e0a75144260e1928e15"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 2.jpg",
        "historicalName": "verified mango disease 2.jpg",
        "sha256": "1da80fe4b68725c7c39d78af8e4a42c65316c4e29d2be3756bef74c02a51d1d0"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 63.jpg",
        "historicalName": "verified mango disease 63.jpg",
        "sha256": "95e7184c9c0c2c0f5e729d16513410470c77e07bc4a787d50c06f79a55977cd8"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 57.jpg",
        "historicalName": "verified mango disease 57.jpg",
        "sha256": "9c9bc581355a1a4bfcd0befe2051135b7903d219df9320587cd88a28c94db5f6"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 7.jpg",
        "historicalName": "verified mango disease 7.jpg",
        "sha256": "0a7326dd87f77caad7cae79d709bd7cdfcb5a0aec2ef96463782b2920b5cb55c"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 55.jpg",
        "historicalName": "verified mango disease 55.jpg",
        "sha256": "d70902b669201181ee27847a13e4bbf684a371c18957b2cbb20b6cfcbf75275a"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 45.jpg",
        "historicalName": "verified mango disease 45.jpg",
        "sha256": "0827d4061caffa787cb8fab501fb3f25f6e4ef40f434ee03787136b0c8f2d4b7"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 54.jpg",
        "historicalName": "verified mango disease 54.jpg",
        "sha256": "4eee0b91cfa3f2c5b66a0f334d2d2e487fd666d7aa10b3ea9f8bdfc9d36dbfea"
    },
    {
        "label": "mango_scab",
        "currentName": "verified mango disease 8.jpg",
        "historicalName": "verified mango disease 8.jpg",
        "sha256": "00239798d21836b06afea3d213aff445d509eabab9cfa5515e687afc20447e97"
    },
    {
        "label": "mango_scab",
        "currentName": "DSC_1014.JPG",
        "historicalName": "DSC_1014.JPG",
        "sha256": "348841118bd2fed6966a5837ddd7dcde55cb053057371a39ce4f051aa5627bdf"
    },
    {
        "label": "mango_scab",
        "currentName": "Scab1.JPG",
        "historicalName": "Scab1.JPG",
        "sha256": "dbf4ee9f20018da3f04c542d9b369403bb8fbf0149b36c14a4363782d68d174e"
    },
    {
        "label": "mango_scab",
        "currentName": "725146731_2449764938863852_7559841505625140608_n.jpg",
        "historicalName": "725146731_2449764938863852_7559841505625140608_n.jpg",
        "sha256": "a86e5a775f479a63fe2528305bc350300435f8b048494a06ae13e1fd78d0d0f5"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_1021.JPG",
        "historicalName": "DSC_1021.JPG",
        "sha256": "9872562f0d3dfe137e0db5edfbaeaa602aef7444aacded8d18ce9de1bfbf5390"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_1019.JPG",
        "historicalName": "DSC_1019.JPG",
        "sha256": "f744b6410bd2ba2897372201bd1179144fbcb6860e11cf2ba083e1fc8e4f1d9e"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_0436.JPG",
        "historicalName": "DSC_0436.JPG",
        "sha256": "7053f37a4c34867f2c77d44ca764def21315667a24ce4cf388586a6be866d458"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_0205.JPG",
        "historicalName": "DSC_0205.JPG",
        "sha256": "0a1ab9392878b940fc828aa7e7659b0ee2897244935f0dd57e8e012384c9cc53"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_0204.JPG",
        "historicalName": "DSC_0204.JPG",
        "sha256": "5e832bc5a93f67dda12f79405fa39e89f7208052bb5ee6e627711967c6ee668c"
    },
    {
        "label": "healthy_fruit",
        "currentName": "DSC_0201 (1).JPG",
        "historicalName": "DSC_0201 (1).JPG",
        "sha256": "84e0237792a86d60a92e12a4da910e7bfe20f60439aff39f3534d6eaeb1c35f4"
    },
    {
        "label": "healthy_fruit",
        "currentName": "727799974_830567853245491_1446073817623999359_n.jpg",
        "historicalName": "727799974_830567853245491_1446073817623999359_n.jpg",
        "sha256": "9e0436c4d16f0517ffb395477f0567ff046fd41d04311ae74c1abb224c496504"
    },
    {
        "label": "healthy_fruit",
        "currentName": "725318273_1696519271391031_8474708639771176175_n.jpg",
        "historicalName": "725318273_1696519271391031_8474708639771176175_n.jpg",
        "sha256": "18c047f48c6deb7b423919369488485c608300cbba79d078428954694073ba4f"
    },
    {
        "label": "healthy_fruit",
        "currentName": "723960758_2381115535716542_3913785565978577803_n.jpg",
        "historicalName": "723960758_2381115535716542_3913785565978577803_n.jpg",
        "sha256": "3fb605dbbc492f8466267cbb8ec52ceede4455c676dc54a66db37b7e909ed644"
    },
    {
        "label": "healthy_fruit",
        "currentName": "723624552_976005712089787_680463173797746853_n.jpg",
        "historicalName": "723624552_976005712089787_680463173797746853_n.jpg",
        "sha256": "bccae50356a84611268b9fd0b5f5bf5517b222ac223d298084b121e169550b64"
    },
    {
        "label": "healthy_fruit",
        "currentName": "723624552_976005712089787_680463173797746853_n_2.jpg",
        "historicalName": "723624552_976005712089787_680463173797746853_n_2.jpg",
        "sha256": "b1c8ddc707402b023f61be78b900eb878af012c14964e1dbf25b7aab060c2310"
    },
    {
        "label": "healthy_fruit",
        "currentName": "719671069_1291039922786074_6951448894192280220_n.jpg",
        "historicalName": "719671069_1291039922786074_6951448894192280220_n.jpg",
        "sha256": "61cabb3193580b025925fe0177dfdf9a62cf7ea19a18df33218c41842349439c"
    }
]


def exact_representatives(rows, protected_hashes=None, preferred_names=None):
    """Choose exact-pixel representatives without relabeling images."""
    protected_hashes = protected_hashes or set()
    preferred_names = preferred_names or {}
    decoded = defaultdict(list)
    for row in rows:
        decoded[row['decodedRgbSha256']].append(row)
    retained = []
    audit = []
    for fingerprint, group in sorted(decoded.items()):
        labels = {row.get('label') for row in group}
        if len(labels) != 1:
            raise RuntimeError('Exact decoded image has conflicting class labels: ' + str([row['sourceRelativePath'] for row in group]))
        protected = {row['sha256'] for row in group if row['sha256'] in protected_hashes}
        if len(protected) > 1:
            raise RuntimeError('Two protected distinct identities have identical decoded pixels; explicit conflict resolution required.')
        chosen = min(group, key=lambda row: (
            row['sha256'] not in protected_hashes,
            Path(row['sourceRelativePath']).name != preferred_names.get(row.get('label')),
            row['sourceRelativePath'],
        ))
        retained.append(dict(chosen))
        if len(group) > 1:
            audit.append({'keptId': chosen['id'], 'copies': portable_rows(group)})
    if not protected_hashes.issubset({row['sha256'] for row in retained}):
        raise RuntimeError('Exact-copy reduction would remove a protected reviewed fingerprint.')
    return retained, audit


def connected_groups(rows, pairs, distinct_pairs, extra_groups):
    """Keep related captures together using reviewed pair decisions."""
    parents = {row['id']: row['id'] for row in rows}

    def root(key):
        """Find the representative of a related-image group."""
        while parents[key] != key:
            parents[key] = parents[parents[key]]
            key = parents[key]
        return key

    def join(left, right):
        """Merge related groups without splitting their identities."""
        first, second = root(left), root(right)
        parents[max(first, second)] = min(first, second)

    audited = {tuple(sorted((pair['firstId'], pair['secondId']))) for pair in pairs}
    distinct = {tuple(sorted(pair)) for pair in distinct_pairs}
    if any(len(pair) != 2 for pair in distinct_pairs) or not distinct.issubset(audited):
        raise ValueError('Distinct decisions must name complete audited pairs.')
    for pair in audited - distinct:
        join(*pair)
    for group in extra_groups:
        if len(group) < 2 or any(key not in parents for key in group):
            raise ValueError('Related-capture groups require at least two known image IDs.')
        for key in group[1:]:
            join(group[0], key)
    return {key: root(key) for key in parents}



def new_similarity_pairs(new_images, other_images, same_pool=False):
    """Review new relationships without redoing the old visual screening."""
    flags = []
    for index, first in enumerate(new_images):
        candidates = other_images[index + 1:] if same_pool else other_images
        for second in candidates:
            if first['id'] == second['id']:
                continue
            distance = (int(first['phash'], 16) ^ int(second['phash'], 16)).bit_count()
            exact = first['sha256'] == second['sha256'] or first['decodedRgbSha256'] == second['decodedRgbSha256']
            if exact or distance <= PHASH_DISTANCE:
                flags.append({'firstId': first['id'], 'secondId': second['id'], 'distance': distance,
                              'exact': exact, 'firstPath': first['sourceRelativePath'],
                              'secondPath': second['sourceRelativePath'], 'secondPool': second['pool']})
    return flags


def assert_disjoint_pools(pools):
    """Reject exact file/pixel identity leakage across scientific pools."""
    for index, (name, first) in enumerate(pools):
        for field in ('sha256', 'decodedRgbSha256'):
            values = [row[field] for row in first]
            if len(values) != len(set(values)):
                raise RuntimeError('Duplicate ' + field + ' inside ' + name)
            for second_name, second in pools[index + 1:]:
                if set(values) & {row[field] for row in second}:
                    raise RuntimeError('Exact overlap between ' + name + ' and ' + second_name)


assert_disjoint_pools([('mango', raw_rows + new_rows), ('OE', approved_oe_rows), ('reporting OOD', held_out_ood_rows)])
new_mango_flags = new_similarity_pairs(new_rows, new_rows, same_pool=True) + new_similarity_pairs(new_rows, raw_rows)
new_cross_flags = new_similarity_pairs(new_rows, approved_oe_rows + held_out_ood_rows)
drifted_mango = [row for row in raw_rows if row.get('referencePhash', row['phash']) != row['phash']]
drifted_oe = [row for row in approved_oe_rows if row.get('referencePhash', row['phash']) != row['phash']]
drifted_reporting = [row for row in held_out_ood_rows if row.get('referencePhash', row['phash']) != row['phash']]
# Revisit every old boundary affected by changed derived fingerprints; do not
# silently reuse an old similarity review under a different representation.
new_mango_flags += new_similarity_pairs(drifted_mango, raw_rows + new_rows)
new_cross_flags += new_similarity_pairs(drifted_mango, approved_oe_rows + held_out_ood_rows)
new_cross_flags += new_similarity_pairs(drifted_oe, raw_rows + new_rows + held_out_ood_rows)
new_cross_flags += new_similarity_pairs(drifted_reporting, raw_rows + new_rows + approved_oe_rows)
new_mango_flags = list({tuple(sorted((row['firstId'], row['secondId']))): row for row in new_mango_flags}.values())
new_cross_flags = list({tuple(sorted((row['firstId'], row['secondId']))): row for row in new_cross_flags}.values())
save_json_once(RUN_DIR / 'duplicate_audit.json', {'newMangoPairs': new_mango_flags, 'newCrossPoolPairs': new_cross_flags,
    'oldMembershipVerified': True, 'historicalScreeningReused': True,
    'changedHistoricalPhashCounts': {'mango': len(drifted_mango), 'oe': len(drifted_oe), 'reporting': len(drifted_reporting)},
    'limits': 'Hash checks do not identify all related captures or establish cultivar/label validity.'})
flagged_ids = {row[key] for row in new_mango_flags + new_cross_flags for key in ('firstId', 'secondId')}
save_contact_sheets([r for r in raw_rows + new_rows + approved_oe_rows + held_out_ood_rows if r['id'] in flagged_ids],
                    RUN_DIR / 'new_similarity_previews', 'V5.1.0 new-photo similarity candidates')
reviewed_hashes = {row['sha256'] for row in REVIEWED_IMAGES}
if not reviewed_hashes.issubset({row['sha256'] for row in raw_rows}):
    raise RuntimeError('A historical reviewed identity is missing.')
save_json_once(RUN_DIR / 'reviewed_images.json', REVIEWED_IMAGES)
print('New mango similarity pairs:', len(new_mango_flags), 'new cross-pool candidates:', len(new_cross_flags))

## Step 6: Review additional labels and related captures

**What:** Record honest new-photo review decisions.

**Why:** Different filenames or hashes do not prove independent captures or correct disease labels.

**How:** Inspect both sheet folders, enter review notes, and identify related specimens/sessions. Similarity alone groups rather than deletes mango photos.

**Expected result:** An immutable grouping review. Unknown label provenance remains unknown.

In [ ]:
NEW_IMAGE_REVIEW_COMPLETE = False
NEW_IMAGE_REVIEW_NOTES = ''  # Describe label sources honestly; do not invent expert verification.
DISTINCT_SIMILAR_PAIRS = []  # [[firstId, secondId], ...], only inspected independent mango pairs.
DISTINCT_PAIR_NOTES = {}  # 'smallerId|largerId': specific inspection reason.
ADDITIONAL_RELATED_GROUPS = []  # Whole known same-specimen/burst/session groups.
NEW_CROSS_POOL_DECISIONS = {}  # 'firstId|secondId': specific reason a flagged pair is distinct.

if not NEW_IMAGE_REVIEW_COMPLETE or not NEW_IMAGE_REVIEW_NOTES.strip():
    raise RuntimeError('Inspect new_image_previews and new_similarity_previews, then complete this review cell.')
known_pairs = {tuple(sorted((row['firstId'], row['secondId']))) for row in new_mango_flags}
distinct = {tuple(sorted(pair)) for pair in DISTINCT_SIMILAR_PAIRS}
if len(distinct) != len(DISTINCT_SIMILAR_PAIRS) or not distinct.issubset(known_pairs):
    raise ValueError('Distinct-pair decisions must identify unique audited mango pairs.')
if set(DISTINCT_PAIR_NOTES) != {'|'.join(pair) for pair in distinct} or any(not str(v).strip() for v in DISTINCT_PAIR_NOTES.values()):
    raise ValueError('Every distinct mango pair needs its own specific reason.')
if set(NEW_CROSS_POOL_DECISIONS) != {row['firstId'] + '|' + row['secondId'] for row in new_cross_flags}:
    raise RuntimeError('Inspect every new cross-pool similarity; record a reason only if the photos are distinct.')
if any(not str(value).strip() for value in NEW_CROSS_POOL_DECISIONS.values()):
    raise ValueError('Empty cross-pool reasons are not approval.')

# Preserve every old related group while connecting new related captures to it.
old_groups = defaultdict(list)
for row in raw_rows:
    old_groups[row['groupId']].append(row['id'])
prior_related = [ids for ids in old_groups.values() if len(ids) > 1]
retained_rows = raw_rows + new_rows
group_assignments = connected_groups(retained_rows, new_mango_flags, DISTINCT_SIMILAR_PAIRS,
                                     prior_related + ADDITIONAL_RELATED_GROUPS)
for row in retained_rows:
    row['groupId'] = group_assignments[row['id']]
save_json_once(RUN_DIR / 'duplicate_review.json', {
    'complete': True, 'notes': NEW_IMAGE_REVIEW_NOTES,
    'distinctPairs': DISTINCT_SIMILAR_PAIRS, 'distinctPairNotes': DISTINCT_PAIR_NOTES,
    'additionalRelatedGroups': ADDITIONAL_RELATED_GROUPS,
    'newCrossPoolDecisions': NEW_CROSS_POOL_DECISIONS, 'groupAssignments': group_assignments,
    'additionalPhotosExpertVerified': 'not established by folder names or hashes',
})
print('New-image grouping approved. Split conflicts still block the next cell.')

## Step 7: Allocate the additional groups near 70/15/15

**What:** Preserve historical membership and assign only new groups.

**Why:** Moving an old test image to training would change the comparison and introduce reuse problems.

**How:** Keep complete related groups together; block connections across old subsets. Preserve the exact seven old validation previews.

**Expected result:** Actual per-class split counts and added-photo assignments, with no claim of an exact percentage.

In [ ]:
def assign_preserving_groups(rows, original_membership, seed, class_names):
    """Keep old membership fixed and allocate only unassigned whole groups."""
    groups = defaultdict(list)
    for row in rows:
        groups[row['groupId']].append(row)
    subsets = ('train', 'val', 'test')
    totals = {label: sum(row['label'] == label for row in rows) for label in class_names}
    targets = {subset: {label: fraction * totals[label] for label in class_names}
               for subset, fraction in zip(subsets, (0.70, 0.15, 0.15))}
    counts = {subset: {label: 0 for label in class_names} for subset in subsets}
    assignments = {}
    for key, members in groups.items():
        fixed = {original_membership[row['sha256']] for row in members if row['sha256'] in original_membership}
        if len(fixed) > 1:
            raise RuntimeError('New grouping connects old training/validation/test photos. Resolve the relationship before training: ' + key)
        if fixed:
            chosen = next(iter(fixed))
            if chosen not in subsets:
                raise ValueError('Unknown historical subset.')
            assignments[key] = chosen
            for row in members:
                counts[chosen][row['label']] += 1
    remaining = sorted(key for key in groups if key not in assignments)
    random.Random(seed).shuffle(remaining)
    remaining.sort(key=lambda key: -len(groups[key]))
    for key in remaining:
        contribution = Counter(row['label'] for row in groups[key])
        scores = []
        for candidate in subsets:
            score = sum(((counts[subset][label] + (contribution[label] if subset == candidate else 0)
                          - targets[subset][label]) / max(targets[subset][label], 1)) ** 2
                        for subset in subsets for label in class_names)
            scores.append(score)
        chosen = subsets[min(range(3), key=lambda index: scores[index])]
        assignments[key] = chosen
        for row in groups[key]:
            counts[chosen][row['label']] += 1
    if any(counts[subset][label] <= 0 for subset in subsets for label in class_names):
        raise RuntimeError('Every class must remain represented in every mango subset.')
    return assignments, counts


original_membership = {row['sha256']: row['subset'] for row in parent_manifest['mango']}
assignments, counts_by_subset = assign_preserving_groups(retained_rows, original_membership, SEED, CLASS_NAMES)
records = []
for row in sorted(retained_rows, key=lambda item: (CLASS_NAMES.index(item['label']), item['id'])):
    saved = dict(row)
    saved['subset'] = assignments[row['groupId']]
    saved['relativePath'] = saved['subset'] + '/' + saved['label'] + '/' + saved['id'] + '__' + Path(saved['sourceRelativePath']).name
    records.append(saved)
preview_hashes = {row['sha256'] for row in parent_manifest['examples']}
named_examples = {row['label']: row for row in records if row['sha256'] in preview_hashes}
if len(named_examples) != 7 or any(row['subset'] != 'val' for row in named_examples.values()):
    raise RuntimeError('The same seven historical preview photos must remain in validation.')
print('Proposed whole-group split:', json.dumps(counts_by_subset, indent=2))
print('Actual fractions:', {subset: sum(r['subset'] == subset for r in records) / len(records) for subset in counts_by_subset})
print('Added-image split:', dict(Counter(row['subset'] for row in records if row['pool'] == 'new_mango')))

## Step 8: Approve and snapshot the split, OE, and reporting inputs

**What:** Store immutable original image copies for this run.

**Why:** Later folder edits or account changes must not silently alter this run's model or evaluation.

**How:** Approve actual counts, copy mango/OE/reporting originals, verify hashes, and save split/preview manifests.

**Expected result:** Frozen dataset, OE, and 995 reporting snapshots; source folders are unchanged.

In [ ]:
SPLIT_REVIEW_COMPLETE = False
SPLIT_REVIEW_NOTES = ''
if not SPLIT_REVIEW_COMPLETE or not SPLIT_REVIEW_NOTES.strip():
    raise RuntimeError('Review printed counts, complete-group assignments, and the reserved original previews before approving.')
verify_image_rows(raw_rows)
verify_image_rows(approved_oe_rows)
verify_image_rows(held_out_ood_rows)
if DATASET_DIR.exists():
    raise RuntimeError('Dataset copies already exist. Do not recreate or overwrite the split in this run.')
DATASET_DIR.mkdir()
for row in records:
    destination = safe_child(DATASET_DIR, row['relativePath'])
    destination.parent.mkdir(parents=True, exist_ok=True)
    with Path(row['path']).open('rb') as source, destination.open('xb') as target:
        shutil.copyfileobj(source, target)
    row['path'] = str(destination)
verify_image_rows(records)

# Preserve OE training originals inside the run so later threshold checks can
# inspect the actual pixels even when the mounted raw input location changes.
oe_dataset = RUN_DIR / 'oe_dataset'
oe_dataset.mkdir()
for row in approved_oe_rows:
    filename = row['id'] + '__' + Path(row['sourceRelativePath']).name
    row['relativePath'] = 'oe_dataset/' + filename
    destination = safe_child(RUN_DIR, row['relativePath'])
    with Path(row['path']).open('rb') as source, destination.open('xb') as target:
        shutil.copyfileobj(source, target)
    row['path'] = str(destination)
verify_image_rows(approved_oe_rows)
save_json_once(RUN_DIR / 'oe_dataset_manifest.json', {'images': portable_rows(approved_oe_rows)})
train_rows = [row for row in records if row['subset'] == 'train']
val_rows = [row for row in records if row['subset'] == 'val']
by_id = {row['id']: row for row in records}
example_rows = [by_id[named_examples[label]['id']] for label in CLASS_NAMES]
if any(row['subset'] != 'val' for row in example_rows):
    raise RuntimeError('All preview identities and their complete groups must remain in validation.')
manifest = {
    'schemaVersion': 1, 'experimentVersion': EXPERIMENT_VERSION, 'classNames': CLASS_NAMES,
    'counts': counts_by_subset, 'splitFractionsTarget': [0.70, 0.15, 0.15],
    'splitPolicy': 'Historical mango membership preserved; new related groups target 70/15/15; same seven validation previews',
    'images': portable_rows(records), 'validationProbabilityOrder': 'images subset=val in listed order',
    'historicalMangoCount': len(raw_rows), 'additionalMangoCount': len(new_rows),
}
save_json_once(RUN_DIR / 'input_manifest.json', manifest)
save_json_once(RUN_DIR / 'class_names.json', CLASS_NAMES)
save_json_once(RUN_DIR / 'split_review.json', {'complete': True, 'notes': SPLIT_REVIEW_NOTES, 'counts': counts_by_subset})
save_json_once(RUN_DIR / 'example_selection.json', {
    'seed': SEED, 'subset': 'val', 'images': portable_rows(example_rows),
    'selectionBeforePredictions': True, 'historicalPreviewSha256': sorted(preview_hashes),
})
with zipfile.ZipFile(RUN_DIR / 'dataset_split.zip', 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for row in records:
        archive.write(row['path'], row['relativePath'])
print('Frozen split copies:', DATASET_DIR)


# Snapshot reporting photos as well; the evaluation notebook needs no raw-folder dependency.
reporting_directory = RUN_DIR / 'reporting_ood'
reporting_directory.mkdir()
for row in held_out_ood_rows:
    row['relativePath'] = 'reporting_ood/' + row['id'] + '__' + Path(row['sourceRelativePath']).name
    destination = safe_child(RUN_DIR, row['relativePath'])
    with Path(row['path']).open('rb') as source, destination.open('xb') as target:
        shutil.copyfileobj(source, target)
    row['path'] = str(destination)
verify_image_rows(held_out_ood_rows)
save_json_once(RUN_DIR / 'reporting_ood_manifest.json', {'images': portable_rows(held_out_ood_rows),
    'count': 995, 'role': 'reporting comparison only; already evaluated in V5; never threshold tuning'})

## Step 9: Inspect the six V4.5.0 augmentation categories

**What:** Define the training transforms and draw previews on the same seven originals.

**Why:** Transforms must retain visible specimen features; illustration copies are not extra training or test photos.

**How:** Use flip, rotation 30 degrees, zoom 0.2, shear 0.2, brightness 0.2, and contrast 0.1. Training batches carry OE IDs for actual exposure counts.

**Expected result:** A V5.1.0 augmentation preview with original, six techniques, and combined columns.

In [ ]:
class RandomShear(tf.keras.layers.Layer):
    def __init__(self, factor=0.2, **kwargs):
        """Initialize this layer and its recorded objective state."""
        super().__init__(**kwargs)
        self.factor = factor

    def call(self, inputs, training=False):
        """Run the layer with explicit training or inference behavior."""
        if not training:
            return inputs
        shape = tf.shape(inputs)
        count = shape[0]
        center_x = tf.cast(shape[2], tf.float32) / 2.0
        center_y = tf.cast(shape[1], tf.float32) / 2.0
        shear_x = tf.random.uniform([count], -self.factor, self.factor)
        shear_y = tf.random.uniform([count], -self.factor, self.factor)
        one, zero = tf.ones_like(shear_x), tf.zeros_like(shear_x)
        transforms = tf.stack((one, shear_x, -shear_x * center_y, shear_y, one, -shear_y * center_x, zero, zero), axis=1)
        return tf.raw_ops.ImageProjectiveTransformV3(
            images=inputs, transforms=transforms, output_shape=shape[1:3],
            interpolation='BILINEAR', fill_mode='NEAREST', fill_value=0.0,
        )


def augmentation_layers(seed):
    """Reproduce the six V4.5.0 training transforms."""
    return [
        tf.keras.layers.RandomFlip('horizontal_and_vertical', seed=seed),
        tf.keras.layers.RandomRotation(30 / 360, fill_mode='nearest', seed=seed + 1),
        tf.keras.layers.RandomZoom(0.2, fill_mode='nearest', seed=seed + 2),
        RandomShear(0.2),
        tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255), seed=seed + 3),
        tf.keras.layers.RandomContrast(CONTRAST_FACTOR, value_range=(0, 255), seed=seed + 4),
    ]


def augment_batch(pixels, layers):
    """Apply training transforms and retain the valid pixel range."""
    output = pixels
    for layer in layers:
        output = layer(output, training=True)
    return tf.clip_by_value(output, 0.0, 255.0)


def image_dataset(rows, labelled=False):
    """Stream one original photo at a time in manifest order."""
    def examples():
        """Yield original image pixels one at a time."""
        for row in rows:
            pixels = load_pixels(row['path']).numpy().astype(np.float32)
            if labelled:
                yield pixels, np.int32(CLASS_NAMES.index(row['label']))
            else:
                yield pixels
    signature = tf.TensorSpec((224, 224, 3), tf.float32)
    if labelled:
        signature = (signature, tf.TensorSpec((), tf.int32))
    dataset = tf.data.Dataset.from_generator(examples, output_signature=signature)
    return dataset.apply(tf.data.experimental.assert_cardinality(len(rows)))


def make_validation_dataset():
    """Batch clean mango validation photos without augmentation."""
    return image_dataset(val_rows, labelled=True).batch(BATCH_SIZE).prefetch(1)


def make_training_dataset(epoch):
    # Every retained mango training identity is presented once per epoch.
    # A separately shuffled OE stream supplies one batch per mango batch and
    # cycles as needed; its count never changes class weights or mango accuracy.
    """Pair one augmented OE batch with each mango batch."""
    mango_order = list(train_rows)
    oe_order = list(approved_oe_rows)
    random.Random(SEED + epoch).shuffle(mango_order)
    random.Random(SEED + 1000 + epoch).shuffle(oe_order)
    mango_layers = augmentation_layers(SEED + 10000 + 10 * epoch)
    oe_layers = augmentation_layers(SEED + 20000 + 10 * epoch)
    mango = image_dataset(mango_order, labelled=True).batch(BATCH_SIZE)
    mango = mango.map(lambda x, y: (augment_batch(x, mango_layers), y), num_parallel_calls=1)
    oe = tf.data.Dataset.zip((image_dataset(oe_order), tf.data.Dataset.from_tensor_slices(
        [approved_oe_rows.index(row) for row in oe_order]))).batch(BATCH_SIZE).repeat()
    oe = oe.map(lambda x, ids: (augment_batch(x, oe_layers), ids), num_parallel_calls=1)
    combined = tf.data.Dataset.zip((mango, oe))
    return combined.map(lambda labelled, outliers: ({'mango': labelled[0], 'oe': outliers[0], 'oe_ids': outliers[1]}, labelled[1]), num_parallel_calls=1).prefetch(1)


fig, axes = plt.subplots(7, 8, figsize=(24, 23))
headings = ['Original', 'Flip', 'Rotation', 'Zoom', 'Shear', 'Brightness', 'Contrast 0.1', 'Combined']
for index, row in enumerate(example_rows):
    pixels = load_pixels(row['path'])[None]
    layers = augmentation_layers(SEED + 30000 + index * 10)
    views = [pixels] + [tf.clip_by_value(layer(pixels, training=True), 0, 255) for layer in layers]
    views.append(augment_batch(pixels, augmentation_layers(SEED + 40000 + index * 10)))
    for axis, heading, view in zip(axes[index], headings, views):
        axis.imshow(np.uint8(view.numpy()[0]))
        axis.set_title(heading)
        axis.set_xticks([])
        axis.set_yticks([])
    axes[index, 0].set_ylabel(row['label'].replace('_', ' '), fontsize=11)
fig.suptitle('V5.1.0 seven-class augmentation preview', fontsize=20)
fig.text(0.5, 0.01, 'Illustration-only validation originals. Training transforms preserve labels; contrast changes light/dark separation. No background replacement or localized lighting.', ha='center', fontsize=11)
fig.tight_layout(rect=(0, 0.025, 1, 0.97))
fig.savefig(FIGURE_DIR / 'augmentation_preview.png', dpi=150)
plt.show()
plt.close(fig)
print('Preview:', FIGURE_DIR / 'augmentation_preview.png')

### Approve the augmentation preview

Inspect every class row before approving the following cell.

In [ ]:
AUGMENTATION_REVIEW_COMPLETE = False
AUGMENTATION_REVIEW_NOTES = ''
if not AUGMENTATION_REVIEW_COMPLETE or not AUGMENTATION_REVIEW_NOTES.strip():
    raise RuntimeError('Inspect all six techniques and combined previews; approve only if important specimen/disease details remain visible.')
save_json_once(RUN_DIR / 'augmentation_review.json', {
    'complete': True, 'notes': AUGMENTATION_REVIEW_NOTES,
    'preview': 'figures/augmentation_preview.png',
    'previewSha256': file_sha256(FIGURE_DIR / 'augmentation_preview.png'),
    'trainingOnly': True,
})

## Step 10: Preflight isolated TensorFlow.js conversion

**What:** Test binary-only converter installation and a disposable model export.

**Why:** Export failures should appear before the fifteen training epochs; converter packages must not replace the training kernel.

**How:** Use an isolated CPU environment, fresh dependency audit, explicit unused-operation advisory records, and a random architecture probe.

**Expected result:** Converter logs and a graph-model probe, or a blocking error. No accuracy is measured.

In [ ]:
CONVERTER_PACKAGES = [
    'pip==26.2.1', 'pip-audit==2.10.1', 'setuptools==81.0.0', 'wheel==0.45.1',
    'tensorflow==2.19.0', 'tf-keras==2.19.0',
    'tensorflow-decision-forests==1.12.0', 'tensorflow-hub==0.16.1',
    'ydf==0.11.0', 'protobuf==5.29.6', 'tensorflowjs==4.22.0',
    'keras==3.15.1', 'numpy==2.1.3', 'Pillow==13.0.0',
    'jax==0.4.35', 'jaxlib==0.4.35', 'flax==0.10.2', 'packaging==23.2',
    'certifi==2026.10.18', 'filelock==4.2.1',
    'idna==3.15', 'msgpack==1.2.3', 'requests==2.33.0',
    'urllib3==2.8.0', 'werkzeug==3.1.9',
]
# TensorFlow.js requires packaging 23.x. TFDF also requires wheel, whose
# compatible 0.45.1 release has a known unused unpack-operation advisory.
def prepare_converter_evidence(run_dir):
    """Preserve this specific failed install before retrying the converter."""
    run_dir = Path(run_dir)
    sealed_names = ('converter_preflight.json', 'plan.json', 'frozen_inputs.json',
                    'training_started.json', 'training_complete.json')
    if any((run_dir / name).exists() for name in sealed_names):
        raise RuntimeError('Converter preflight or training is already frozen; preserve the existing run.')

    evidence = run_dir / 'converter_preflight'
    archived = None
    if evidence.exists():
        expected_logs = {'environment-setup.log', 'package-installation.log'}
        existing = {path.name for path in evidence.iterdir()}
        if existing != expected_logs or not all((evidence / name).is_file() for name in expected_logs):
            raise RuntimeError('Existing converter evidence is not the reviewed installation failure; inspect it before retrying.')
        install_log = (evidence / 'package-installation.log').read_text(encoding='utf-8')
        conflict_details = ('No solution found', 'wheel==0.46.3', 'packaging>=24.0', 'packaging==23.2')
        if not all(detail in install_log for detail in conflict_details):
            raise RuntimeError('The existing converter failure has a different cause; preserve its logs and diagnose it first.')
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        archived = evidence.with_name('converter_preflight.failed-' + stamp + '-' + uuid.uuid4().hex[:8])
        evidence.rename(archived)
        print('Failed converter logs preserved:', archived)

    evidence.mkdir(exist_ok=False)
    if archived is not None:
        preserved_logs = evidence / 'prior_failed_attempt'
        preserved_logs.mkdir()
        for path in sorted(archived.iterdir()):
            shutil.copy2(path, preserved_logs / path.name)
        save_json_once(evidence / 'retry-provenance.json', {
            'reason': 'Corrected incompatible wheel 0.46.3 and packaging 23.2 pins before training.',
            'previousAttempt': archived.relative_to(run_dir).as_posix(),
            'preservedLogs': {
                path.relative_to(run_dir).as_posix(): {'sha256': file_sha256(path), 'bytes': path.stat().st_size}
                for path in sorted(preserved_logs.iterdir())
            },
            'retryPackages': CONVERTER_PACKAGES,
        })
    return evidence


converter_evidence = prepare_converter_evidence(RUN_DIR)
# A new subprocess environment keeps the failed environment and active kernel intact.
converter_environment_root = Path('/content/mangoleaf-v510-converter') / (run_id + '-' + uuid.uuid4().hex[:8])
converter_environment_root.parent.mkdir(parents=True, exist_ok=True)
CONVERTER_ENVIRONMENT = dict(os.environ)
for key in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV', 'TF_USE_LEGACY_KERAS'):
    CONVERTER_ENVIRONMENT.pop(key, None)
CONVERTER_ENVIRONMENT.update({'PYTHONNOUSERSITE': '1', 'CUDA_VISIBLE_DEVICES': '-1', 'TF_CPP_MIN_LOG_LEVEL': '2'})


def converter_run(command, log_name):
    """Run isolated converter tooling and retain its complete log."""
    result = subprocess.run(command, env=CONVERTER_ENVIRONMENT, capture_output=True, text=True, check=False)
    with (converter_evidence / log_name).open('x', encoding='utf-8') as handle:
        handle.write(result.stdout + '\n' + result.stderr)
    if result.returncode != 0:
        print((result.stdout + '\n' + result.stderr)[-4000:])
        raise RuntimeError('Converter preparation failed. Inspect: ' + str(converter_evidence / log_name))
    return result


converter_run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', str(converter_environment_root)], 'environment-setup.log')
CONVERTER_PYTHON = converter_environment_root / 'bin/python'
CONVERTER_COMMAND = converter_environment_root / 'bin/tensorflowjs_converter'
converter_run([
    sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(CONVERTER_PYTHON),
    '--only-binary=:all:', *CONVERTER_PACKAGES,
], 'package-installation.log')
converter_run([str(CONVERTER_PYTHON), '-m', 'pip', 'check'], 'pip-check.log')
freeze_result = converter_run([str(CONVERTER_PYTHON), '-m', 'pip', 'freeze', '--all'], 'freeze.log')
(converter_evidence / 'converter-runtime-lock.txt').write_text(freeze_result.stdout, encoding='utf-8')
audit_path = converter_evidence / 'audit.json'
audit = subprocess.run([
    str(CONVERTER_PYTHON), '-m', 'pip_audit', '--format=json',
    '--output', str(audit_path), '--progress-spinner=off',
], env=CONVERTER_ENVIRONMENT, capture_output=True, text=True, check=False)
(converter_evidence / 'audit.log').write_text(audit.stdout + '\n' + audit.stderr)
report = read_json(audit_path) if audit_path.is_file() else {}
dependencies = report.get('dependencies', [])
findings = [row for row in dependencies if row.get('vulns')]
skipped = [row for row in dependencies if row.get('skip_reason')]

def reviewed_converter_operations(finding, platform_name, os_name):
    """Match only recorded unused operations in the Linux binary-only converter."""
    if platform_name != 'linux' or os_name != 'posix':
        return None
    reviews = {
        ('setuptools', '81.0.0'): [
            ({'GHSA-h35f-9h28-mq5c', 'CVE-2026-59890', 'PYSEC-2026-3447', 'BIT-setuptools-2026-59890'},
             'GHSA-h35f-9h28-mq5c',
             'Source-distribution build/publish operation is absent. TensorFlow Hub imports pkg_resources, removed in newer setuptools. Installation uses only published binary wheels through uv.'),
            ({'GHSA-xmj3-9gf7-h52w'}, 'GHSA-xmj3-9gf7-h52w',
             'Windows legacy Wheel.install_as_egg path is absent: converter runs on Linux and uv installs binary wheels; no setuptools legacy wheel installer or namespace-package conversion is called.'),
        ],
        ('wheel', '0.45.1'): [
            ({'GHSA-8rrh-rw8j-w5fx', 'CVE-2026-24049'}, 'GHSA-8rrh-rw8j-w5fx',
             'wheel.cli.unpack/RECORD permission extraction is absent. TFDF requires wheel and TensorFlow.js requires packaging 23.x; uv installs binary wheels independently.'),
        ],
    }
    available = reviews.get((canonicalize_name(finding.get('name', '')), finding.get('version')), [])
    result = []
    for vulnerability in finding.get('vulns', []):
        identities = {vulnerability.get('id'), *vulnerability.get('aliases', [])}
        matched = next(((advisory, reason) for aliases, advisory, reason in available
                        if identities and identities <= aliases), None)
        if matched is None:
            return None
        advisory, reason = matched
        result.append({'package': finding['name'], 'version': finding['version'], 'advisory': advisory,
            'status': 'unresolved advisory; affected operation absent in this converter',
            'reason': reason, 'source': 'https://github.com/advisories/' + advisory})
    return result or None


operation_reviews = []
blocking = []
for finding in findings:
    reviews = reviewed_converter_operations(finding, sys.platform, os.name)
    if reviews is None:
        blocking.append(finding)
    else:
        operation_reviews.extend(reviews)
audit_allowed = bool(dependencies) and not skipped and not blocking and (
    audit.returncode == 0 and not findings or audit.returncode == 1 and bool(operation_reviews)
)
save_json_once(converter_evidence / 'audit-summary.json', {
    'findings': findings, 'skipped': skipped, 'returnCode': audit.returncode,
    'operationReviews': operation_reviews, 'executionAllowed': bool(audit_allowed),
    'status': 'passed' if not findings and audit_allowed else 'unresolved',
})
if not audit_allowed:
    raise RuntimeError('Converter audit contains unresolved execution findings or incomplete collection. Read ' + str(audit_path))
if operation_reviews:
    print('Known unused packaging-operation advisories remain visible:', audit_path)


def convert_saved_model(saved_model_path, export_path, log_path):
    """Convert and check a plain unquantized TensorFlow.js graph."""
    result = subprocess.run([
        str(CONVERTER_COMMAND), '--input_format=tf_saved_model',
        '--output_format=tfjs_graph_model', '--signature_name=serving_default',
        '--saved_model_tags=serve', str(saved_model_path), str(export_path),
    ], env=CONVERTER_ENVIRONMENT, capture_output=True, text=True, check=False)
    Path(log_path).write_text(result.stdout + '\n' + result.stderr, encoding='utf-8')
    if result.returncode != 0:
        raise RuntimeError('Converter failed. Inspect ' + str(log_path))
    graph = read_json(export_path / 'model.json')
    if graph.get('format') != 'graph-model' or not graph.get('weightsManifest'):
        raise RuntimeError('Converter did not produce a TensorFlow.js graph model.')
    for group in graph['weightsManifest']:
        for relative in group['paths']:
            safe_child(export_path, relative).stat()
        if any(weight.get('quantization') is not None for weight in group['weights']):
            raise RuntimeError('This experiment requires unquantized float32 export.')


probe_backbone = tf.keras.applications.MobileNetV3Large(
    input_shape=(224, 224, 3), include_top=False, weights=None, include_preprocessing=True,
)
probe_inputs = tf.keras.Input((224, 224, 3), dtype=tf.float32, name='image')
probe_features = probe_backbone(probe_inputs, training=False)
probe_pooled = tf.keras.layers.GlobalAveragePooling2D()(probe_features)
probe_outputs = tf.keras.layers.Dense(7, activation='softmax')(probe_pooled)
probe_model = tf.keras.Model(probe_inputs, probe_outputs)
probe_model.export(converter_evidence / 'probe_saved_model', verbose=False)
convert_saved_model(converter_evidence / 'probe_saved_model', converter_evidence / 'probe_tfjs', converter_evidence / 'conversion.log')
save_json_once(RUN_DIR / 'converter_preflight.json', {
    'passed': True, 'trainingEpochs': 0, 'weights': 'random architecture probe',
    'tfjsFormat': 'graph-model', 'quantized': False,
    'browserProbabilityParity': 'pending actual browser comparison',
    'pins': CONVERTER_PACKAGES,
})
del probe_model, probe_backbone
tf.keras.utils.set_random_seed(SEED)
print('Fresh architecture conversion completed; no accuracy measured.')

## Step 11: Initialize a new seven-class ImageNet classifier

**What:** Build fresh mango-specific weights and balanced mango class weights.

**Why:** This experiment tests fresh training rather than inheriting V4.5.0 or V5.0.0 mango weights.

**How:** Use 224x224 RGB float32 pixels in 0-255, internal MobileNet preprocessing, frozen ImageNet features, and a newly initialized head.

**Expected result:** Initial weight identity and training-only class weights; no previous MangoLeaf model is read.

In [ ]:
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=(224, 224, 3), include_top=False,
    weights='imagenet', include_preprocessing=True,
)
base_model.trainable = False
inputs = tf.keras.Input((224, 224, 3), dtype=tf.float32, name='image')
features = base_model(inputs, training=False)
pooled = tf.keras.layers.GlobalAveragePooling2D(name='global_average_pool')(features)
pooled = tf.keras.layers.Dropout(0.2, name='head_dropout')(pooled)
logits = tf.keras.layers.Dense(len(CLASS_NAMES), name='class_logits')(pooled)
outputs = tf.keras.layers.Activation('softmax', name='class_probabilities')(logits)
model = tf.keras.Model(inputs, outputs, name='mangoleaf_v5_1_0_seven_class')
logits_model = tf.keras.Model(inputs, logits, name='mangoleaf_v5_1_0_logits')

train_labels = np.array([CLASS_NAMES.index(row['label']) for row in train_rows])
weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=train_labels)
class_weight = {index: float(value) for index, value in enumerate(weights)}
save_json_once(RUN_DIR / 'class_weights.json', {CLASS_NAMES[index]: value for index, value in class_weight.items()})
initial_digest = hashlib.sha256()
for array in model.get_weights():
    initial_digest.update(str(array.shape).encode())
    initial_digest.update(array.tobytes())
save_json_once(RUN_DIR / 'initial_weights.json', {
    'sha256': initial_digest.hexdigest(), 'source': 'ImageNet backbone plus newly initialized head', 'seed': SEED,
})
print('Training-only weights:', {CLASS_NAMES[index]: value for index, value in class_weight.items()})
model.summary()

### Check the OE objective and plain-model save/reload

These synthetic probes check implementation wiring, not dataset accuracy.

In [ ]:
def uniform_exposure_loss(logits):
    """Encourage uniform seven-class scores for unsupported training photos."""
    return -tf.reduce_mean(tf.nn.log_softmax(logits, axis=-1), axis=-1)


toy_logits = tf.Variable([[4.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]])
before_loss = float(tf.reduce_mean(uniform_exposure_loss(toy_logits)))
before_maximum = float(tf.reduce_max(tf.nn.softmax(toy_logits)))
with tf.GradientTape() as tape:
    toy_loss = tf.reduce_mean(uniform_exposure_loss(toy_logits))
toy_gradient = tape.gradient(toy_loss, toy_logits)
if toy_gradient is None or not np.isfinite(toy_gradient.numpy()).all() or not np.any(np.abs(toy_gradient.numpy()) > 0):
    raise RuntimeError('OE uniform-target loss has no valid nonzero gradient.')
toy_logits.assign_sub(0.1 * toy_gradient)
after_loss = float(tf.reduce_mean(uniform_exposure_loss(toy_logits)))
after_maximum = float(tf.reduce_max(tf.nn.softmax(toy_logits)))
if not after_loss < before_loss or not after_maximum < before_maximum:
    raise RuntimeError('OE toy update did not reduce concentration as expected.')
plain_probe_path = RUN_DIR / 'runtime_preflight/plain_classifier_probe.keras'
model.save(plain_probe_path)
plain_probe = tf.keras.models.load_model(plain_probe_path, compile=False, safe_mode=True)
synthetic_image = tf.zeros((1, 224, 224, 3), dtype=tf.float32)
np.testing.assert_allclose(model(synthetic_image, training=False), plain_probe(synthetic_image, training=False), rtol=1e-4, atol=1e-5)
if plain_probe.output_shape[-1] != 7:
    raise RuntimeError('The ordinary inference classifier must have seven outputs.')
save_json_once(RUN_DIR / 'runtime_preflight/oe_mathematical_probe.json', {
    'trainedMangoEpochs': 0, 'uniformTargetClassCount': 7,
    'beforeLoss': before_loss, 'afterLoss': after_loss,
    'beforeMaximumProbability': before_maximum, 'afterMaximumProbability': after_maximum,
    'finiteNonzeroGradient': True, 'plainClassifierRoundTripPassed': True,
    'accuracyOrOodRejectionMeasured': False,
})
del plain_probe, toy_logits
print('OE gradient and plain inference-model round-trip probes passed; no mango epochs trained.')

## Step 12: Freeze this experiment before fitting

**What:** Seal inputs, recipe, runtime, preview selections, and the fixed confidence rule.

**Why:** A version label alone cannot identify an experimental attempt.

**How:** Hash every copied file and record the V5.1.0 configuration. OE and reporting photos never choose class weights or calibration.

**Expected result:** Frozen artifact hashes and configuration identity; test predictions remain pending.

In [ ]:
plan = {
    'schemaVersion': 1, 'experimentVersion': EXPERIMENT_VERSION, 'modelVersion': MODEL_VERSION,
    'classNames': CLASS_NAMES, 'seed': SEED, 'imageSize': list(IMAGE_SIZE), 'batchSize': BATCH_SIZE,
    'initialWeights': 'ImageNet MobileNetV3Large; no prior MangoLeaf model',
    'splitPolicy': manifest['splitPolicy'], 'splitFractionsTarget': [0.70, 0.15, 0.15],
    'stage1': {'epochs': STAGE1_EPOCHS, 'learningRate': STAGE1_LEARNING_RATE, 'backboneFrozen': True},
    'stage2': {'epochs': STAGE2_EPOCHS, 'learningRate': STAGE2_LEARNING_RATE, 'batchNormalizationFrozen': True},
    'modelSelection': 'Lowest clean unweighted mango validation loss in each stage; final checkpoint from stage 2',
    'augmentation': {'flip': 'horizontal_and_vertical', 'rotationDegrees': 30, 'zoom': 0.2, 'shear': 0.2, 'brightness': 0.2, 'contrast': CONTRAST_FACTOR},
    'oe': {
        'lossWeight': OE_LOSS_WEIGHT, 'target': 'uniform over seven mango classes',
        'loss': 'negative mean log-softmax over classes; expectation on approved OE training photos',
        'objective': 'mean(class_weight * mango cross-entropy) + lossWeight * mean(OE uniform cross-entropy)',
        'weightRationale': '0.5 is a starting research hyperparameter, not inferred from photo count or a proven MangoLeaf optimum',
        'sampling': 'One independently shuffled OE batch per mango batch, cycling as needed; reshuffle every epoch',
        'approvedCount': len(approved_oe_rows), 'accuracyIncludesOE': False,
        'classWeightsIncludeOE': False, 'temperatureIncludesOE': False,
    },
    'classWeightRule': 'Balanced weights from mango training labels only',
    'phashDistance': PHASH_DISTANCE, 'preprocessing': PREPROCESSING,
    'temperatureRule': 'Validation mango negative log likelihood only; divide classifier logits once',
    'heldOutOodCount': len(held_out_ood_rows), 'expectedOodCount': EXPECTED_OOD_IMAGE_COUNT,
    'validationExamples': [row['id'] for row in example_rows],
    'testPredictionsRun': False, 'confidenceThreshold': CONFIDENCE_THRESHOLD,
    'rejectionRule': 'maximum calibrated probability >= 0.70 only',
    'referenceFiles': REFERENCE_SHA256,
    'evaluationLimits': 'Old test and OOD reporting photos have prior evaluation use; not independent field evidence',
}
save_json_once(RUN_DIR / 'plan.json', plan)

frozen_paths = [
    'plan.json', 'input_manifest.json', 'dataset_split.zip', 'class_names.json',
    'mango_inventory.json', 'new_mango_inventory.json', 'oe_inventory.json',
    'oe_dataset_manifest.json', 'held_out_ood_inventory.json', 'reporting_ood_manifest.json',
    'reviewed_images.json', 'duplicate_audit.json', 'duplicate_review.json', 'phash_drift_audit.json',
    'split_review.json', 'example_selection.json', 'augmentation_review.json',
    'figures/augmentation_preview.png', 'runtime.json', 'converter_preflight.json',
    'class_weights.json', 'initial_weights.json',
]
frozen_paths += [row['relativePath'] for row in approved_oe_rows + held_out_ood_rows]
frozen_paths += ['dataset/' + row['relativePath'] for row in records]
frozen_paths += [path.relative_to(RUN_DIR).as_posix() for directory in ('runtime_preflight', 'converter_preflight', 'reference-inputs', 'inventory_cache_v2')
                 for path in sorted((RUN_DIR / directory).rglob('*')) if path.is_file()]
frozen_paths += [path.relative_to(RUN_DIR).as_posix() for path in sorted(RUN_DIR.glob('*_input_selection_v2.json'))]
frozen_paths += [path.relative_to(RUN_DIR).as_posix() for path in sorted(RUN_DIR.glob('step4_repair_*.json'))]
frozen_paths = sorted(set(frozen_paths))
frozen = {
    'configurationHash': object_sha256(plan),
    'artifacts': {relative: {'sha256': file_sha256(safe_child(RUN_DIR, relative)), 'bytes': safe_child(RUN_DIR, relative).stat().st_size}
                  for relative in frozen_paths},
}
save_json_once(RUN_DIR / 'frozen_inputs.json', frozen)


def verify_receipt_artifacts(receipt):
    """Check every sealed artifact before consuming its evidence."""
    for relative, expected in receipt['artifacts'].items():
        artifact = safe_child(RUN_DIR, relative)
        if artifact.stat().st_size != expected['bytes'] or file_sha256(artifact) != expected['sha256']:
            raise RuntimeError('Sealed artifact changed: ' + relative)


print('Frozen configuration:', frozen['configurationHash'])
print('Frozen artifacts:', len(frozen['artifacts']))

## Step 13: Train five head epochs and ten fine-tuning epochs

**What:** Optimize V4.5.0's supervised mango and uniform OE objectives.

**Why:** Mango-only validation loss selects checkpoints; OE photos must not inflate mango accuracy.

**How:** Train at 0.001 then 0.00001 with BatchNormalization frozen in stage 2. Count OE IDs only after successful optimizer updates.

**Expected result:** Fifteen actual epoch records, two selected checkpoints, and measured OE exposure, without test predictions.

In [ ]:
def verify_initial_training_state(classifier, balanced_weights, weight_by_index):
    # Verify the live objects consumed by fit, not only the frozen JSON files.
    # A failed rerun of model creation must not silently replace this run's head.
    """Check the live initial weights and mango-only class weights."""
    initial_digest = hashlib.sha256()
    for array in classifier.get_weights():
        initial_digest.update(str(array.shape).encode())
        initial_digest.update(array.tobytes())
    if initial_digest.hexdigest() != read_json(RUN_DIR / 'initial_weights.json')['sha256']:
        raise RuntimeError('Live initial model weights differ from the frozen identity; create a new run.')

    counts = np.array([sum(row['label'] == label for row in train_rows) for label in CLASS_NAMES], dtype=np.float64)
    if np.any(counts <= 0):
        raise ValueError('Every mango class needs at least one training photo.')
    expected = len(train_rows) / (len(CLASS_NAMES) * counts)
    live = np.asarray(balanced_weights, dtype=np.float64)
    sealed = read_json(RUN_DIR / 'class_weights.json')
    if set(sealed) != set(CLASS_NAMES) or set(weight_by_index) != set(range(len(CLASS_NAMES))):
        raise RuntimeError('Class-weight identities differ from the agreed seven classes.')
    sealed_vector = np.array([sealed[label] for label in CLASS_NAMES], dtype=np.float64)
    mapping_vector = np.array([weight_by_index[index] for index in range(len(CLASS_NAMES))], dtype=np.float64)
    for values in [live, sealed_vector, mapping_vector]:
        if values.shape != expected.shape or not np.isfinite(values).all() or not np.allclose(values, expected, rtol=1e-12, atol=0):
            raise RuntimeError('Live or sealed class weights differ from fresh balanced mango counts; create a new run.')


class OETrainer(tf.keras.Model):
    def __init__(self, classifier_logits, balanced_weights, oe_weight):
        """Initialize this layer and its recorded objective state."""
        super().__init__(name='mangoleaf_oe_training_wrapper')
        self.classifier_logits = classifier_logits
        self.balanced_weights = tf.constant(balanced_weights, dtype=tf.float32)
        self.oe_weight = tf.constant(oe_weight, dtype=tf.float32)
        self.exposure_counts = tf.Variable(tf.zeros(len(approved_oe_rows), dtype=tf.int64), trainable=False)
        self.total_tracker = tf.keras.metrics.Mean(name='loss')
        self.mango_tracker = tf.keras.metrics.Mean(name='mango_loss')
        self.oe_tracker = tf.keras.metrics.Mean(name='oe_loss')
        self.accuracy_tracker = tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')

    def call(self, inputs, training=False):
        # Keras uses call() for symbolic shape/build checks before fit. The
        # custom train_step still computes both mango and OE objectives.
        """Run the layer with explicit training or inference behavior."""
        pixels = inputs['mango'] if isinstance(inputs, dict) else inputs
        return self.classifier_logits(pixels, training=training)

    @property
    def metrics(self):
        """Expose the training and mango-only metric trackers."""
        return [self.total_tracker, self.mango_tracker, self.oe_tracker, self.accuracy_tracker]

    def train_step(self, data):
        """Update the mango and OE objective and count successful exposure."""
        inputs, labels, _ = tf.keras.utils.unpack_x_y_sample_weight(data)
        with tf.GradientTape() as tape:
            mango_logits = self.classifier_logits(inputs['mango'], training=True)
            outlier_logits = self.classifier_logits(inputs['oe'], training=True)
            original_losses = tf.keras.losses.sparse_categorical_crossentropy(labels, mango_logits, from_logits=True)
            per_photo_weighted = original_losses * tf.gather(self.balanced_weights, labels)
            supervised_loss = tf.reduce_mean(per_photo_weighted)
            # A uniform target uses every output class equally. OE has no
            # fabricated Healthy Leaf label and no additional output class.
            per_outlier_loss = uniform_exposure_loss(outlier_logits)
            oe_loss = tf.reduce_mean(per_outlier_loss)
            total_loss = supervised_loss + self.oe_weight * oe_loss
        variables = self.classifier_logits.trainable_variables
        gradients = tape.gradient(total_loss, variables)
        pairs = [(gradient, variable) for gradient, variable in zip(gradients, variables) if gradient is not None]
        if len(pairs) != len(variables):
            raise RuntimeError('A trainable classifier variable has no gradient.')
        self.optimizer.apply_gradients(pairs)
        # Count only OE examples consumed by a successful optimizer update.
        self.exposure_counts.scatter_add(tf.IndexedSlices(
            tf.ones_like(inputs['oe_ids'], dtype=tf.int64), tf.cast(inputs['oe_ids'], tf.int32)))
        self.total_tracker.update_state(total_loss, sample_weight=tf.cast(tf.shape(labels)[0], tf.float32))
        self.mango_tracker.update_state(per_photo_weighted)
        self.oe_tracker.update_state(per_outlier_loss)
        self.accuracy_tracker.update_state(labels, tf.nn.softmax(mango_logits))
        return {metric.name: metric.result() for metric in self.metrics}

    def test_step(self, data):
        """Measure clean mango validation without OE or class weighting."""
        pixels, labels, _ = tf.keras.utils.unpack_x_y_sample_weight(data)
        logits = self.classifier_logits(pixels, training=False)
        # Validation loss is clean, unweighted mango cross-entropy. It never
        # includes OE uncertainty loss or balanced training class weights.
        clean_losses = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
        self.total_tracker.update_state(clean_losses)
        self.mango_tracker.update_state(clean_losses)
        self.oe_tracker.update_state(0.0)
        self.accuracy_tracker.update_state(labels, tf.nn.softmax(logits))
        return {metric.name: metric.result() for metric in self.metrics}


if (RUN_DIR / 'training_started.json').exists():
    raise RuntimeError('This run already started fitting. Preserve an interrupted attempt; create a new run to retrain.')
if object_sha256(plan) != frozen['configurationHash']:
    raise RuntimeError('Plan changed after freeze.')
live_recipe = {'flip': 'horizontal_and_vertical', 'rotationDegrees': 30, 'zoom': 0.2, 'shear': 0.2, 'brightness': 0.2, 'contrast': CONTRAST_FACTOR}
if plan['augmentation'] != live_recipe or plan['oe']['lossWeight'] != OE_LOSS_WEIGHT:
    raise RuntimeError('Augmentation or OE settings changed after freeze; create a new run.')
if plan['stage1'] != {'epochs': STAGE1_EPOCHS, 'learningRate': STAGE1_LEARNING_RATE, 'backboneFrozen': True} or plan['stage2'] != {'epochs': STAGE2_EPOCHS, 'learningRate': STAGE2_LEARNING_RATE, 'batchNormalizationFrozen': True}:
    raise RuntimeError('Stage settings changed after freeze; create a new run.')
if portable_rows(records) != manifest['images'] or read_json(RUN_DIR / 'input_manifest.json') != manifest:
    raise RuntimeError('Live mango identities/order differ from the frozen manifest.')
if portable_rows(train_rows) != [row for row in manifest['images'] if row['subset'] == 'train'] or portable_rows(val_rows) != [row for row in manifest['images'] if row['subset'] == 'val']:
    raise RuntimeError('Live training or validation rows differ from the frozen mango subsets.')
if portable_rows(approved_oe_rows) != read_json(RUN_DIR / 'oe_dataset_manifest.json')['images']:
    raise RuntimeError('Live OE training identities/order differ from the frozen manifest.')
verify_receipt_artifacts(frozen)
verify_image_rows(records)
verify_image_rows(approved_oe_rows)
verify_image_rows(held_out_ood_rows)
verify_initial_training_state(model, weights, class_weight)
if logits_model.get_layer('class_logits') is not model.get_layer('class_logits') or logits_model.get_layer(base_model.name) is not base_model:
    raise RuntimeError('The live training logits are not connected to the verified plain classifier.')
save_json_once(RUN_DIR / 'training_started.json', {
    'startedAt': datetime.now(timezone.utc).isoformat(), 'configurationHash': frozen['configurationHash'],
})

training_history = []
completed_epochs = 0
cumulative_exposure = np.zeros(len(approved_oe_rows), dtype=np.int64)
all_successful_exposure = np.zeros_like(cumulative_exposure)
selected_exposure = np.zeros_like(cumulative_exposure)
for stage, epoch_count, learning_rate in (
    (1, STAGE1_EPOCHS, STAGE1_LEARNING_RATE),
    (2, STAGE2_EPOCHS, STAGE2_LEARNING_RATE),
):
    if stage == 2:
        base_model.trainable = True
        for layer in base_model.layers:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
    trainer = OETrainer(logits_model, weights, OE_LOSS_WEIGHT)
    trainer.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), jit_compile=False)
    best_loss, best_epoch = math.inf, None
    checkpoint = RUN_DIR / f'stage{stage}_best.weights.h5'
    for stage_epoch in range(epoch_count):
        print(f'Stage {stage}; overall epoch {completed_epochs + 1}/{STAGE1_EPOCHS + STAGE2_EPOCHS}')
        history = trainer.fit(
            make_training_dataset(completed_epochs), validation_data=make_validation_dataset(),
            epochs=1, shuffle=False, verbose=2,
        )
        metrics = {name: float(values[-1]) for name, values in history.history.items()}
        metrics['learning_rate'] = float(tf.keras.backend.get_value(trainer.optimizer.learning_rate))
        if not all(math.isfinite(value) for value in metrics.values()):
            raise RuntimeError('Non-finite training metric; do not export this incomplete run.')
        completed_epochs += 1
        row = {
            'stage': stage, 'epoch': completed_epochs, **metrics,
            'mangoTrainingCount': len(train_rows),
            'oeBatches': math.ceil(len(train_rows) / BATCH_SIZE),
            'uniqueOeExposedSoFar': int(np.count_nonzero(cumulative_exposure + trainer.exposure_counts.numpy())),
        }
        training_history.append(row)
        save_json_once(RUN_DIR / f'epoch_{completed_epochs:02d}.json', row)
        if metrics['val_loss'] < best_loss:
            best_loss, best_epoch = metrics['val_loss'], completed_epochs
            model.save_weights(checkpoint)
            selected_exposure = cumulative_exposure + trainer.exposure_counts.numpy()
    cumulative_exposure += trainer.exposure_counts.numpy()
    all_successful_exposure += trainer.exposure_counts.numpy()
    model.load_weights(checkpoint)
    # Stage 2 inherits only the selected stage 1 checkpoint's exposure.
    if stage == 1:
        cumulative_exposure = selected_exposure.copy()
    save_json_once(RUN_DIR / f'stage{stage}_selection.json', {
        'stage': stage, 'selectedEpoch': best_epoch, 'validationLoss': best_loss,
        'criterion': 'clean unweighted mango validation cross-entropy',
    })

save_json_once(RUN_DIR / 'oe_exposure.json', {
    'availableUniquePhotos': len(approved_oe_rows),
    'uniquePhotosExposed': int(np.count_nonzero(all_successful_exposure)),
    'selectedCheckpointUniquePhotosExposed': int(np.count_nonzero(selected_exposure)),
    'successfulExamples': int(all_successful_exposure.sum()),
    'selectedCheckpointSuccessfulExamples': int(selected_exposure.sum()),
    'selectedCheckpointCountsBySha256': {row['sha256']: int(value) for row, value in zip(approved_oe_rows, selected_exposure)},
    'note': 'Selected stage 2 inherits selected stage 1 exposure; discarded updates count only in full-run totals.',
})
selected_model = model
selected_model.save(RUN_DIR / 'model_selected.keras')
save_json_once(RUN_DIR / 'training_history.json', training_history)
verify_receipt_artifacts(frozen)
verify_image_rows(records)
verify_image_rows(approved_oe_rows)
fitted_paths = list(frozen['artifacts']) + [
    'frozen_inputs.json', 'training_started.json', 'model_selected.keras', 'training_history.json', 'oe_exposure.json',
    'stage1_best.weights.h5', 'stage2_best.weights.h5', 'stage1_selection.json', 'stage2_selection.json',
]
save_json_once(RUN_DIR / 'training_state.json', {
    'status': 'fitted; calibration/export pending', 'configurationHash': frozen['configurationHash'],
    'completedEpochs': completed_epochs, 'testPredictionsRun': False,
    'artifacts': {relative: {'sha256': file_sha256(safe_child(RUN_DIR, relative)), 'bytes': safe_child(RUN_DIR, relative).stat().st_size}
                  for relative in fitted_paths},
})
print('Selected stage 2 checkpoint:', read_json(RUN_DIR / 'stage2_selection.json'))

## Step 14: Fit a new validation-only temperature

**What:** Calibrate this newly trained model's confidence.

**Why:** An earlier model's calibration value cannot describe the new weights.

**How:** Minimize validation negative log likelihood, fold temperature exactly once, and check unchanged highest-scoring classes.

**Expected result:** New calibrated model and temperature records; test and OOD inputs are excluded.

In [ ]:
verify_receipt_artifacts(read_json(RUN_DIR / 'training_state.json'))
if (RUN_DIR / 'calibration_complete.json').exists():
    raise RuntimeError('Calibration is already sealed. Read its artifacts rather than refitting or folding temperature twice.')


def predict_records(classifier, rows):
    """Predict a recorded image sequence without changing its order."""
    return classifier.predict(image_dataset(rows).batch(BATCH_SIZE).prefetch(1), verbose=0)


logits_predictor = tf.keras.Model(selected_model.inputs, selected_model.get_layer('class_logits').output)
validation_logits = predict_records(logits_predictor, val_rows).astype(np.float64)
validation_labels = np.array([CLASS_NAMES.index(row['label']) for row in val_rows])


def validation_nll(temperature):
    """Measure clean validation negative log likelihood at one temperature."""
    scaled = validation_logits / temperature
    shifted = scaled - scaled.max(axis=1, keepdims=True)
    log_probabilities = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    return float(-np.mean(log_probabilities[np.arange(len(validation_labels)), validation_labels]))


calibration = minimize_scalar(validation_nll, bounds=(0.25, 20.0), method='bounded')
if not calibration.success or not math.isfinite(calibration.fun):
    raise RuntimeError('Validation-only temperature fit failed.')
temperature = float(calibration.x) if calibration.fun < validation_nll(1.0) else 1.0
calibrated_model = tf.keras.models.clone_model(selected_model)
calibrated_model.set_weights(selected_model.get_weights())
head = calibrated_model.get_layer('class_logits')
head_weights, head_bias = head.get_weights()
head.set_weights([head_weights / temperature, head_bias / temperature])
probabilities = predict_records(calibrated_model, val_rows)
original_probabilities = predict_records(selected_model, val_rows)
if not np.array_equal(original_probabilities.argmax(axis=1), probabilities.argmax(axis=1)):
    raise RuntimeError('Temperature folding changed a validation class prediction.')
if not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-5):
    raise RuntimeError('Invalid calibrated validation probabilities.')
calibrated_model.save(RUN_DIR / 'model_calibrated.keras')
np.save(RUN_DIR / 'validation_logits.npy', validation_logits)
np.save(RUN_DIR / 'validation_probabilities.npy', probabilities)
reloaded = tf.keras.models.load_model(RUN_DIR / 'model_calibrated.keras', compile=False, safe_mode=True)
np.testing.assert_allclose(probabilities, predict_records(reloaded, val_rows), rtol=1e-4, atol=1e-5)
temperature_record = {
    'temperature': temperature, 'temperatureFoldedIntoWeights': True, 'foldCount': 1,
    'fitSubset': 'val', 'fitIncludesOE': False, 'fitIncludesOOD': False,
    'validationCount': len(val_rows), 'beforeNll': validation_nll(1.0), 'afterNll': validation_nll(temperature),
    'searchBounds': [0.25, 20.0], 'keepUnitTemperatureUnlessImproved': True,
    'validationProbabilityOrder': 'input_manifest.json images subset=val in listed order',
}
save_json_once(RUN_DIR / 'temperature.json', temperature_record)
calibration_paths = ['model_calibrated.keras', 'validation_logits.npy', 'validation_probabilities.npy', 'temperature.json']
save_json_once(RUN_DIR / 'calibration_complete.json', {
    'configurationHash': frozen['configurationHash'],
    'artifacts': {relative: {'sha256': file_sha256(RUN_DIR / relative), 'bytes': (RUN_DIR / relative).stat().st_size}
                  for relative in calibration_paths},
})
print('Validation-fitted temperature:', temperature)

## Step 15: Export the new calibrated model

**What:** Create unquantized TensorFlow.js candidate files.

**Why:** The application requires a plain inference graph, not the OE training wrapper.

**How:** Export calibrated weights through the isolated converter; retain all conversion logs and hashes.

**Expected result:** A candidate graph model; browser probability parity and adoption remain pending.

In [ ]:
verify_receipt_artifacts(read_json(RUN_DIR / 'calibration_complete.json'))
destination = RUN_DIR / 'tfjs_model'
if destination.exists() or (RUN_DIR / 'export_complete.json').exists():
    raise RuntimeError('Export output already exists. Preserve it; never silently overwrite a sealed or partial attempt.')
attempt = RUN_DIR / 'export_attempts' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])
attempt.mkdir(parents=True, exist_ok=False)
saved_model_path = attempt / 'saved_model'
exported_path = attempt / 'tfjs_model'
calibrated_model.export(saved_model_path, verbose=False)
convert_saved_model(saved_model_path, exported_path, attempt / 'conversion.log')
configuration = {
    'modelVersion': MODEL_VERSION, 'experimentVersion': EXPERIMENT_VERSION,
    'status': 'experimental candidate; fixed 70% confidence reporting pending',
    'classNames': CLASS_NAMES, 'classes': CLASS_NAMES,
    'inputSize': [224, 224], 'inputRange': [0, 255], 'inputDtype': 'float32',
    'preprocessing': PREPROCESSING, 'temperature': temperature_record['temperature'],
    'temperatureFoldedIntoWeights': True, 'quantized': False,
    'browserParityStatus': 'pending actual browser comparison before adoption',
    'trainingObjective': 'mango supervision plus uniform-target Outlier Exposure',
}
save_json_once(exported_path / 'model_config.json', configuration)
shutil.copytree(exported_path, destination)
save_json_once(RUN_DIR / 'export_complete.json', {
    'format': 'graph-model', 'quantized': False,
    'attempt': attempt.relative_to(RUN_DIR).as_posix(),
    'calibratedModelSha256': file_sha256(RUN_DIR / 'model_calibrated.keras'),
    'artifacts': {path.relative_to(RUN_DIR).as_posix(): {'sha256': file_sha256(path), 'bytes': path.stat().st_size}
                  for path in sorted(destination.rglob('*')) if path.is_file()},
})
print('Candidate export:', destination)

## Step 16: Generate new Grad-CAM on the same seven originals

**What:** Draw real connected true-class gradients and calibrated prediction captions.

**Why:** The old layout can be reused, but its old heatmaps and scores cannot describe V5.1.0.

**How:** Use exactly the augmentation preview identities, the selected trained model, and original/map/overlay columns.

**Expected result:** V5.1.0 Grad-CAM with visible mistakes or missing positive signal; no field-accuracy claim.

In [ ]:
def connected_grad_cam(classifier, batch, target_index):
    """Trace true-class logits through the actual trained head."""
    backbone = next((layer for layer in classifier.layers if isinstance(layer, tf.keras.Model)), None)
    if backbone is None:
        raise ValueError('A connected MobileNet backbone is required for Grad-CAM.')
    head_layers = classifier.layers[classifier.layers.index(backbone) + 1:]
    dense = next((layer for layer in head_layers if isinstance(layer, tf.keras.layers.Dense)), None)
    if dense is None or dense.get_config()['activation'] != 'linear':
        raise ValueError('Grad-CAM requires the real linear classifier logits.')
    features = backbone(batch, training=False)
    if features.shape.rank != 4:
        raise ValueError('Backbone output must be a spatial feature map.')
    with tf.GradientTape() as tape:
        tape.watch(features)
        output = features
        for layer in head_layers:
            output = layer(output, training=False)
            if layer is dense:
                logits = output
                break
        score = logits[:, target_index]
    gradients = tape.gradient(score, features)
    if gradients is None or not np.isfinite(gradients.numpy()).all():
        raise RuntimeError('Grad-CAM gradients are missing or non-finite.')
    weights = tf.reduce_mean(gradients, axis=(1, 2))
    heatmap = tf.maximum(tf.reduce_sum(features[0] * weights[0], axis=-1), 0)
    maximum = float(tf.reduce_max(heatmap))
    if maximum > 0:
        heatmap = heatmap / maximum
    resized = tf.image.resize(heatmap[..., None], batch.shape[1:3], method='bilinear')[..., 0]
    return resized.numpy(), maximum > 0, backbone.name, dense.name


figure_manifest = {
    'experimentVersion': EXPERIMENT_VERSION, 'subset': 'val',
    'illustrationsEnterTrainingOrScoring': False,
    'selectionBeforePredictions': True, 'examples': [],
}
fig, axes = plt.subplots(7, 3, figsize=(13, 25))
for index, row in enumerate(example_rows):
    pixels = load_pixels(row['path']).numpy()
    batch = tf.convert_to_tensor(pixels[None], dtype=tf.float32)
    probabilities = calibrated_model(batch, training=False).numpy()[0]
    predicted_index = int(np.argmax(probabilities))
    target_index = CLASS_NAMES.index(row['label'])
    heatmap, has_signal, backbone_name, logits_name = connected_grad_cam(selected_model, batch, target_index)
    colored = plt.colormaps['jet'](heatmap)[..., :3]
    overlay = np.clip(0.65 * pixels / 255 + 0.35 * colored, 0, 1) if has_signal else pixels / 255
    headings = [
        'True: ' + row['label'].replace('_', ' '),
        'True-class Grad-CAM' if has_signal else 'No positive Grad-CAM signal',
        'Predicted: ' + CLASS_NAMES[predicted_index].replace('_', ' ') + f"\nCalibrated confidence: {probabilities[predicted_index]:.1%}",
    ]
    for axis, view, heading in zip(axes[index], [pixels / 255, heatmap, overlay], headings):
        axis.imshow(view, cmap='jet' if view.ndim == 2 else None, vmin=0, vmax=1)
        axis.set_title(heading, fontsize=11)
        axis.axis('off')
    figure_manifest['examples'].append({
        'id': row['id'], 'label': row['label'], 'sourceRelativePath': row['sourceRelativePath'],
        'sha256': row['sha256'], 'groupId': row['groupId'], 'subset': row['subset'],
        'gradCamTargetClass': row['label'], 'predictedClass': CLASS_NAMES[predicted_index],
        'calibratedConfidence': float(probabilities[predicted_index]),
        'backboneLayer': backbone_name, 'logitsLayer': logits_name, 'hasPositiveHeatmap': has_signal,
    })
fig.suptitle('V5.1.0 seven-class validation illustration Grad-CAM', fontsize=18)
fig.text(0.5, 0.01, 'Warmer colors show stronger positive contribution to the true-class logit. Heatmaps do not prove correct diagnosis; predictions and confidence remain visible.', ha='center', fontsize=10)
fig.tight_layout(rect=(0, 0.025, 1, 0.98))
fig.savefig(FIGURE_DIR / 'grad_cam.png', dpi=150)
plt.show()
plt.close(fig)

## Step 17: Plot the recorded training curves

**What:** Show actual mango accuracy and mango loss over fifteen epochs.

**Why:** Weighted augmented training loss and clean validation loss have different definitions.

**How:** Plot unsmoothed values, stage boundary, and both learning rates; keep OE/total losses in the records.

**Expected result:** New training_curves.png and figure manifest from actual measurements.

In [ ]:
history = read_json(RUN_DIR / 'training_history.json')
if [row['epoch'] for row in history] != list(range(1, STAGE1_EPOCHS + STAGE2_EPOCHS + 1)):
    raise RuntimeError('Training history must contain all fifteen epochs in order.')
epochs = [row['epoch'] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(19, 6.5))
axes[0].plot(epochs, [row['accuracy'] for row in history], 'o-', label='Training mango')
axes[0].plot(epochs, [row['val_accuracy'] for row in history], 's-', label='Validation mango')
axes[0].set_title('Accuracy: stage 1 (frozen base), then stage 2 (fine-tuning)')
axes[0].set_ylabel('Accuracy')
axes[1].plot(epochs, [row['mango_loss'] for row in history], 'o-', label='Training mango (weighted)')
axes[1].plot(epochs, [row['val_loss'] for row in history], 's-', label='Validation mango (unweighted)')
axes[1].set_title('Mango cross-entropy: weighted training and clean validation')
axes[1].set_ylabel('Loss')
for axis in axes:
    axis.axvline(STAGE1_EPOCHS + 0.5, color='gray', linestyle='--', label='Stage 2 begins')
    axis.set_xlabel('Epoch')
    axis.grid(alpha=0.25)
    axis.legend()
    axis.text(0.02, 0.03, 'Stage 1: LR 0.001 | Stage 2: LR 0.00001', transform=axis.transAxes, fontsize=10)
fig.suptitle('V5.1.0 training history: 5 epochs at 1e-3, then 10 epochs at 1e-5', fontsize=18)
fig.text(0.5, 0.012, 'Accuracy excludes OE. Training mango loss includes class weights and augmentation; validation is clean and unweighted. OE and total loss are recorded separately in training_history.json.', ha='center', fontsize=10)
fig.tight_layout(rect=(0, 0.055, 1, 0.94))
fig.savefig(FIGURE_DIR / 'training_curves.png', dpi=160)
plt.show()
plt.close(fig)
figure_manifest['figures'] = {
    name: {'sha256': file_sha256(FIGURE_DIR / name), 'bytes': (FIGURE_DIR / name).stat().st_size}
    for name in ('augmentation_preview.png', 'grad_cam.png', 'training_curves.png')
}
figure_manifest['lossPanel'] = 'Weighted augmented mango training loss versus clean unweighted mango validation loss; not total OE objective'
save_json_once(RUN_DIR / 'figure_manifest.json', figure_manifest)

## Step 18: Seal the completed training handoff

**What:** Verify and seal this run's model, snapshots, and results.

**Why:** An interrupted or changed attempt must not be presented as completed.

**How:** Check all fifteen epochs, frozen inputs, calibration, export, reporting snapshots, and evidence hashes.

**Expected result:** training_complete.json plus the exact run path for the evaluation notebook; no deployment.

In [ ]:
if (RUN_DIR / 'training_complete.json').exists():
    raise RuntimeError('This run is already sealed; read the receipt without overwriting it.')
for receipt_name in ('training_state.json', 'calibration_complete.json', 'export_complete.json'):
    verify_receipt_artifacts(read_json(RUN_DIR / receipt_name))
fitted = read_json(RUN_DIR / 'training_state.json')
if fitted['completedEpochs'] != STAGE1_EPOCHS + STAGE2_EPOCHS:
    raise RuntimeError('All fifteen epochs must finish before sealing.')
verify_image_rows(records)
verify_image_rows(approved_oe_rows)
verify_image_rows(held_out_ood_rows)
required = [
    'frozen_inputs.json', 'training_started.json', 'training_state.json',
    'model_selected.keras', 'model_calibrated.keras', 'temperature.json', 'oe_exposure.json',
    'validation_logits.npy', 'validation_probabilities.npy', 'training_history.json',
    'calibration_complete.json', 'export_complete.json',
    'stage1_best.weights.h5', 'stage2_best.weights.h5', 'stage1_selection.json', 'stage2_selection.json',
    'figure_manifest.json', 'figures/augmentation_preview.png', 'figures/grad_cam.png', 'figures/training_curves.png',
]
required += list(frozen['artifacts'])
required += [f'epoch_{epoch:02d}.json' for epoch in range(1, STAGE1_EPOCHS + STAGE2_EPOCHS + 1)]
required += [path.relative_to(RUN_DIR).as_posix() for path in sorted((RUN_DIR / 'tfjs_model').rglob('*')) if path.is_file()]
required = sorted(set(required))
if any(PurePosixPath(relative).suffix == '.gsheet' for relative in required):
    raise ValueError('Google Sheets shortcuts are not source CSV files; use actual reviewed JSON/CSV evidence.')
receipt = {
    'schemaVersion': 1, 'experimentVersion': EXPERIMENT_VERSION, 'modelVersion': MODEL_VERSION,
    'status': 'training/export complete; experimental candidate; fixed-confidence final evaluation and browser parity pending',
    'completedAt': datetime.now(timezone.utc).isoformat(), 'classNames': CLASS_NAMES,
    'configurationHash': fitted['configurationHash'],
    'inputManifestSha256': file_sha256(RUN_DIR / 'input_manifest.json'),
    'calibratedTemperature': temperature_record['temperature'],
    'temperatureFoldedIntoWeights': True, 'foldedTemperature': True,
    'testPredictionsRun': False, 'browserParityPassed': False,
    'completedEpochs': fitted['completedEpochs'], 'selectedCheckpoint': read_json(RUN_DIR / 'stage2_selection.json'),
    'heldOutOodCount': len(held_out_ood_rows), 'oeTrainingCount': len(approved_oe_rows),
    'artifacts': {relative: {'sha256': file_sha256(safe_child(RUN_DIR, relative)), 'bytes': safe_child(RUN_DIR, relative).stat().st_size}
                  for relative in required},
}
save_json_once(RUN_DIR / 'training_complete.json', receipt)
print('Finished standalone v5.1.0 training run:', RUN_DIR)
print('Next: MangoLeaf_AI_Evaluate_v5_1_0.ipynb using this exact run. No threshold selection is needed.')

## Sources and interpretation limits

- [TensorFlow transfer learning](https://www.tensorflow.org/tutorials/images/transfer_learning): ImageNet initialization and frozen BatchNormalization during fine-tuning.
- [Outlier Exposure](https://arxiv.org/abs/1812.04606): auxiliary unsupported inputs; coefficient 0.5 is an experimental setting.
- [Keras RandomContrast](https://keras.io/api/layers/preprocessing_layers/image_augmentation/random_contrast/): 0.1 samples contrast factors 0.9-1.1.
- [Wheel advisory](https://github.com/advisories/GHSA-8rrh-rw8j-w5fx) and [setuptools advisory](https://github.com/advisories/GHSA-h35f-9h28-mq5c): retained converter findings are explicitly scoped to unused operations, never reported as patched. Any new finding blocks the preflight.
- [Setuptools Windows legacy-install advisory](https://github.com/advisories/GHSA-xmj3-9gf7-h52w): the Linux binary-only converter does not call the affected installer. The training/evaluation environment instead uses patched setuptools 84.1.0. All raw converter findings remain in its audit output.

The old mango test and 995 unsupported photos have prior evaluation use. Reuse supports a recorded comparison, not a fresh independent field test. The 71 new labels have only the provenance documented in their review; folder names and hashes do not establish expert review or Carabao cultivar. Grad-CAM does not validate diagnosis. Application adoption, phone reliability, browser probability parity, and scientific claims require separate review.